# 🧠 TikTok Ads - ML Analytics Engine

**Advanced Machine Learning & Statistical Analysis for Product Performance**

---

## 🔄 ARSITEKTUR DRY (Don't Repeat Yourself)

```
┌─────────────────────────────────────────────────────────────┐
│                   tiktok_ads_core.py                        │
│                 (SINGLE SOURCE OF TRUTH)                    │
│  ├── load_product_names()    - Load dari Excel              │
│  ├── load_data()             - Load dari Database           │
│  ├── analyze_products()      - ML Scoring & Kategorisasi    │
│  ├── mann_kendall_test()     - Statistical Trend Test       │
│  ├── calculate_momentum()    - Momentum Analysis            │
│  ├── generate_report_md()    - Generate Markdown Report     │
│  └── generate_html()         - Convert to HTML              │
└─────────────────────────────────────────────────────────────┘
                    ↓                    ↓
    ┌──────────────────────┐    ┌──────────────────────┐
    │  generate_report.py  │    │  tiktok_ads_ml.ipynb │
    │  (Command Line/MCP)  │    │  (Exploration/Viz)   │
    │  - Otomatis          │    │  - Interactive       │
    │  - Tanpa visualisasi │    │  - Dengan plot       │
    └──────────────────────┘    └──────────────────────┘
```

## 📊 Analysis Methods:
| Method | Purpose |
|--------|--------|
| **Mann-Kendall Test** | Trend statistical significance |
| **Momentum Analysis** | Recent vs historical performance |
| **Coefficient of Variation** | Consistency analysis |
| **Composite Scoring** | ROI 30%, Profit 25%, Momentum 20%, Consistency 15%, Trend 10% |

## 🎯 Output Categories:
| Category | Criteria | Action |
|----------|----------|--------|
| 🟢 **LANJUTKAN** | Score ≥60, ROI ≥2x, Profit >0 | Scale up budget |
| 🟡 **PANTAU** | Score 40-60, inconsistent | Monitor 7 days |
| 🔴 **HENTIKAN** | Score <40, ROI <1x, loss | Stop immediately |

---

## ⚡ QUICK START

**Opsi 1: Via MCP (Copilot)**
```
@workspace Analisis performa iklan TikTok dan generate laporan
```

**Opsi 2: Via Command Line**
```bash
cd notebooks
python generate_report.py
```

**Opsi 3: Via Notebook (interaktif)**
Jalankan cell di bawah ini satu per satu.

---

In [ ]:
# ============================================================
# CORE MODULE IMPORT (DRY - Single Source of Truth)
# ============================================================
# Semua fungsi analisis ada di tiktok_ads_core.py
# Notebook ini untuk EKSPLORASI & VISUALISASI INTERAKTIF
# Script generate_report.py untuk OTOMASI (via MCP)

from tiktok_ads_core import (
    # Data loading
    load_product_names, 
    load_data, 
    get_product_display_name,
    
    # Statistical Analysis (Professional ML Methods)
    mann_kendall_test,      # Non-parametric trend test
    linear_trend_analysis,  # Slope, R², p-value
    calculate_momentum,     # Recent vs historical (70/30)
    coefficient_of_variation, # Consistency measurement
    
    # Main Analysis
    analyze_products,       # Full ML scoring
    
    # Report generation
    generate_report_md,
    generate_html,
    run_full_analysis,
    
    # Quick access helpers
    get_top_products,
    get_stop_products,
    get_monitor_products,
    summary_stats,
    
    # Config
    TENANT_ID,
    DB_PATH,
    EXCEL_PRODUCT_REF,
    OUTPUT_DIR
)

# Additional imports for visualization
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

print("✅ Core module loaded with professional ML methods!")
print(f"   Database: {DB_PATH}")
print(f"   Tenant: {TENANT_ID}")
print("\n📊 Available Statistical Methods:")
print("   - Mann-Kendall Trend Test (non-parametric)")
print("   - Linear Regression Analysis (slope, R², p-value)")
print("   - Momentum Analysis (70/30 split)")
print("   - Coefficient of Variation (consistency)")

In [ ]:
# ============================================================
# QUICK ANALYSIS (Using Core Module - Same as MCP/Script)
# ============================================================
# Gunakan fungsi dari core untuk analisis cepat
# HASIL DIJAMIN SAMA dengan generate_report.py dan MCP!

# Load data
product_map = load_product_names()
df = load_data()

# Analyze with full ML methods
products = analyze_products(df, product_map)

# Summary
stats_result = summary_stats(df)
print("\n" + "=" * 70)
print("📊 RINGKASAN PERFORMA (Identik dengan Script & MCP)")
print("=" * 70)
print(f"   Total Investment: Rp {stats_result['total_cost']:,.0f}")
print(f"   Total Revenue: Rp {stats_result['total_revenue']:,.0f}")
print(f"   Total Profit: Rp {stats_result['total_profit']:,.0f}")
print(f"   Overall ROI: {stats_result['roi']:.2f}x")
print(f"   Total Produk: {stats_result['total_products']}")

print("\n🎯 DISTRIBUSI REKOMENDASI:")
lanjut = len([p for p in products if 'LANJUTKAN' in p['category']])
pantau = len([p for p in products if 'PANTAU' in p['category']])
henti = len([p for p in products if 'HENTIKAN' in p['category']])
print(f"   🟢 LANJUTKAN: {lanjut}")
print(f"   🟡 PANTAU: {pantau}")
print(f"   🔴 HENTIKAN: {henti}")

# Show sample with full statistics
print("\n📈 SAMPLE PRODUK DENGAN STATISTIK LENGKAP:")
print("-" * 70)
for p in products[:3]:
    print(f"\n{p['product_name']}")
    print(f"   ROI: {p['roi']:.2f}x | Profit: Rp {p['profit']:,}")
    print(f"   Mann-Kendall: {p['trend']} (p={p['trend_p_value']:.4f}, τ={p['trend_tau']:.4f})")
    print(f"   Linear Reg: {p['lr_direction']} (R²={p['lr_r_squared']:.4f}, slope={p['lr_slope']:.6f})")
    print(f"   Momentum: {p['momentum']} ({p['momentum_pct']:+.1f}%)")
    print(f"   Consistency: {p['consistency']} (CV={p['cv']}%)")
    print(f"   Composite Score: {p['score']} → {p['category']}")

In [ ]:
# ============================================================
# CELL EKSPLORASI: Analisis Data Produk yang Tersedia
# ============================================================
import pandas as pd
import sqlite3
import os

# Path files (relatif dari folder notebooks)
EXCEL_PATH = "ads_data/Tiktoksellercenter_batchedit_20260114_basic_information_template.xlsx"
DB_PATH = "../backend/config/databases/yumna_bertigamart.db"

# 1. Cek struktur Excel (referensi nama produk)
print("=" * 70)
print("📊 ANALISIS FILE EXCEL (Referensi Nama Produk)")
print("=" * 70)
try:
    excel_df = pd.read_excel(EXCEL_PATH)
    print(f"Kolom tersedia: {list(excel_df.columns)}")
    print(f"Total baris: {len(excel_df)}")
    print("\nSample data:")
    display(excel_df.head(3))
except Exception as e:
    print(f"Error membaca Excel: {e}")

# 2. Cek data TiktokAdsCreativeData
print("\n" + "=" * 70)
print("📊 ANALISIS DATABASE - TiktokAdsCreativeData")
print("=" * 70)
conn = sqlite3.connect(DB_PATH)
ads_df = pd.read_sql_query("SELECT * FROM TiktokAdsCreativeData LIMIT 5", conn)
print(f"Kolom: {list(ads_df.columns)}")
print(f"\nSample productId:")
print(ads_df[['productId', 'campaignName', 'videoTitle']].head())

# 3. Cek apakah ada data di TiktokProduct
print("\n" + "=" * 70)
print("📊 ANALISIS DATABASE - TiktokProduct")
print("=" * 70)
try:
    product_df = pd.read_sql_query("SELECT * FROM TiktokProduct LIMIT 5", conn)
    print(f"Total products: {len(pd.read_sql_query('SELECT COUNT(*) as c FROM TiktokProduct', conn)['c'].iloc[0])}")
    print(f"Kolom: {list(product_df.columns)}")
    display(product_df.head())
except Exception as e:
    print(f"Tabel TiktokProduct kosong atau tidak ada: {e}")

conn.close()

# 4. Summary
print("\n" + "=" * 70)
print("🔍 KESIMPULAN AWAL")
print("=" * 70)

📊 ANALISIS FILE EXCEL (Referensi Nama Produk)
Kolom tersedia: ['product_id', 'sale_platforms', 'category', 'brand', 'product_name', 'product_description', 'tts_product_url', 'toko_product_url']
Total baris: 114

Sample data:


,product_id,sale_platforms,category,brand,product_name,product_description,tts_product_url,toko_product_url
0,V3,Basic_Information,NaN,NaN,NaN,NaN,NaN,NaN
1,ID Produk,Platform,Kategori,Merek,Nama Produk,Deskripsi Produk,Tautan (TikTok Shop),Tautan (Tokopedia)
2,Wajib,Wajib,Wajib,Opsional,Wajib,Wajib,Wajib,Wajib



📊 ANALISIS DATABASE - TiktokAdsCreativeData
Kolom: ['id', 'tenantId', 'uploadBatchId', 'periodStart', 'periodEnd', 'campaignId', 'campaignName', 'productId', 'creativeType', 'videoTitle', 'videoId', 'tiktokAccount', 'postingTime', 'status', 'authorizationType', 'cost', 'ordersSku', 'costPerOrder', 'grossRevenue', 'roi', 'impressions', 'clicks', 'ctr', 'conversionRate', 'watchRate2s', 'watchRate6s', 'watchRate25pct', 'watchRate50pct', 'watchRate75pct', 'watchRate100pct', 'currency', 'createdAt', 'updatedAt']

Sample productId:
             productId               campaignName  \
0  1729732509484222628   Aktif sejak 11 Juni 2025   
1  1730228114064049316  Aktif sejak 10 april 2025   
2  1730228114064049316  Aktif sejak 10 april 2025   
3  1730448467452069028    Aktif sejak 18 Okt 2025   
4  1730228114064049316  Aktif sejak 10 april 2025   

                                          videoTitle  
0                                               None  
1  Inner hijab dagu premium nusseyba s

In [ ]:
# ============================================================
# CELL: Analisis Jenis Materi Iklan (creativeType)
# ============================================================
import pandas as pd
import sqlite3

DB_PATH = "../backend/config/databases/yumna_bertigamart.db"
conn = sqlite3.connect(DB_PATH)

# Cek kolom creativeType
ads_df = pd.read_sql_query("""
    SELECT * FROM TiktokAdsCreativeData WHERE tenantId = 'yumna_bertigamart'
""", conn)
conn.close()

print("=" * 70)
print("📊 ANALISIS JENIS MATERI IKLAN (creativeType)")
print("=" * 70)

# Lihat nilai unik
print("\n🔹 Nilai unik creativeType:")
print(ads_df['creativeType'].unique())

print("\n🔹 Distribusi jenis materi iklan:")
type_dist = ads_df['creativeType'].value_counts()
print(type_dist)

# Analisis performa per tipe
print("\n" + "=" * 70)
print("💰 PERFORMA PER JENIS MATERI IKLAN")
print("=" * 70)
perf_by_type = ads_df.groupby('creativeType').agg({
    'grossRevenue': ['sum', 'mean'],
    'cost': ['sum', 'mean'],
    'roi': 'mean',
    'ordersSku': 'sum',
    'productId': 'nunique',
    'id': 'count'
}).round(2)
perf_by_type.columns = ['total_revenue', 'avg_revenue', 'total_cost', 'avg_cost', 
                        'avg_roi', 'total_orders', 'n_products', 'n_creatives']
display(perf_by_type)

print(f"\n⚡ INSIGHT PENTING:")
for creative_type in ads_df['creativeType'].unique():
    subset = ads_df[ads_df['creativeType'] == creative_type]
    icon = "🎬" if creative_type == "Video" else "🎴"
    print(f"\n{icon} {creative_type}:")
    print(f"   - {len(subset):,} creatives, {subset['productId'].nunique()} produk")
    print(f"   - Total Revenue: Rp {subset['grossRevenue'].sum():,.0f}")
    print(f"   - Total Cost: Rp {subset['cost'].sum():,.0f}")
    print(f"   - Avg ROI: {subset['roi'].mean():.2f}")

print("\n" + "=" * 70)
print("⚠️ KESIMPULAN: Analisis ML harus DIPISAHKAN per jenis materi iklan!")
print("   • Video: Cocok untuk content marketing, engagement")
print("   • Kartu Produk: Cocok untuk direct conversion, catalog sales")
print("=" * 70)

📊 ANALISIS JENIS MATERI IKLAN (creativeType)

🔹 Nilai unik creativeType:
['Kartu produk' 'Video']

🔹 Distribusi jenis materi iklan:
creativeType
Video           83315
Kartu produk      344
Name: count, dtype: int64

💰 PERFORMA PER JENIS MATERI IKLAN


,total_revenue,avg_revenue,total_cost,avg_cost,avg_roi,total_orders,n_products,n_creatives
creativeType,,,,,,,,
Kartu produk,2.545307e+09,7399149.1,92798074.0,269761.84,25.58,17906,92,344
Video,4.889832e+09,58690.9,921854536.0,11064.69,37.31,45866,2699,83315



⚡ INSIGHT:

Kartu produk:
   - 344 creatives, 92 produk
   - Total Revenue: Rp 2,545,307,292
   - Total Cost: Rp 92,798,074
   - Avg ROI: 25.58

Video:
   - 83315 creatives, 2699 produk
   - Total Revenue: Rp 4,889,832,318
   - Total Cost: Rp 921,854,536
   - Avg ROI: 37.31


In [ ]:
# ============================================================
# CELL: Analisis Detail Mapping Product ID ke Nama
# ============================================================
import pandas as pd
import sqlite3

# 1. Baca Excel dengan benar (skip 3 header rows)
EXCEL_PATH = "../docs/ads_data/Tiktoksellercenter_batchedit_20260114_basic_information_template.xlsx"
excel_df = pd.read_excel(EXCEL_PATH, skiprows=3)
excel_df.columns = ['product_id', 'sale_platforms', 'category', 'brand', 
                    'product_name', 'product_description', 'tts_url', 'toko_url']

print("=" * 70)
print("📊 DATA EXCEL - REFERENSI NAMA PRODUK")
print("=" * 70)
print(f"Total produk di Excel: {len(excel_df)}")
print(f"\nSample data:")
display(excel_df[['product_id', 'product_name', 'category']].head(10))

# 2. Cek product IDs di database Ads
DB_PATH = "../backend/config/databases/yumna_bertigamart.db"
conn = sqlite3.connect(DB_PATH)
ads_products = pd.read_sql_query("""
    SELECT DISTINCT productId 
    FROM TiktokAdsCreativeData 
    WHERE tenantId = 'yumna_bertigamart'
""", conn)
print(f"\n📊 Product IDs di Database Ads: {len(ads_products)}")

# 3. Cek TiktokProduct table
tiktok_products = pd.read_sql_query("SELECT productId, name FROM TiktokProduct", conn)
print(f"📊 Products di TiktokProduct table: {len(tiktok_products)}")
conn.close()

# 4. Cek overlap/matching
excel_df['product_id'] = excel_df['product_id'].astype(str)
ads_products['productId'] = ads_products['productId'].astype(str)

ads_ids = set(ads_products['productId'].tolist())
excel_ids = set(excel_df['product_id'].tolist())
tiktok_ids = set(tiktok_products['productId'].astype(str).tolist()) if len(tiktok_products) > 0 else set()

matched = ads_ids.intersection(excel_ids)
not_in_excel = ads_ids - excel_ids

print(f"\n" + "=" * 70)
print("🔍 ANALISIS MATCHING")
print("=" * 70)
print(f"✅ Product IDs yang MATCH (di Ads & di Excel): {len(matched)}")
print(f"❌ Product IDs di Ads tapi TIDAK di Excel: {len(not_in_excel)}")
print(f"📦 Product IDs di TiktokProduct: {len(tiktok_ids)}")

if len(matched) > 0:
    print(f"\n✅ Sample matched IDs: {list(matched)[:5]}")
if len(not_in_excel) > 0:
    print(f"\n❌ Sample unmatched IDs: {list(not_in_excel)[:5]}")

📊 DATA EXCEL - REFERENSI NAMA PRODUK
Total produk di Excel: 111

Sample data:


,product_id,product_name,category
0,Tidak dapat diedit,Panjang nama produk harus kurang dari 255 kara...,Tidak dapat diedit
1,1733055736054318244,Nusseyba - Bahira Handshock - Handshock Jempol...,Handsock (601345)
2,1733204152982930596,Nusseyba - Dzikra Top - Atasan Basic Batwing D...,Baju Olahraga Muslim (839048)
3,1732476736733283492,Nusseyba - Khala Dress - Gamis Wanita Simple D...,Gamis (936072)
4,1731521780235338916,Nusseyba - Hana Khimar Bandana - Jersey Khiban...,Khimar (601309)
5,1732318700109726884,Nusseyba - Bilya Skirt - Rok Olahraga Dry Fit ...,Rok Olahraga (963720)
6,1731365288154334372,Nusseyba - Arabella Khimar Bandana - Jersey Kh...,Khimar (601309)
7,1730927911385138340,Nusseyba - Tazkiya Khimar Bandana - Khimar Han...,Khimar (601309)
8,1732491544228955300,Nusseyba - Mika Bag - Tas Tenteng Mika Transpa...,Tote Bag (903176)
9,1732148829314057380,Nusseyba - Hilly Khimar Bandana - Khiban Dry F...,Khimar (601309)



📊 Product IDs di Database Ads: 2702
📊 Products di TiktokProduct table: 92

🔍 ANALISIS MATCHING
✅ Product IDs yang MATCH (di Ads & di Excel): 93
❌ Product IDs di Ads tapi TIDAK di Excel: 2609
📦 Product IDs di TiktokProduct: 92

✅ Sample matched IDs: ['1731901363564741796', '1729996619938236580', '1729943550833100964', '1732148580878222500', '1730395889207772324']

❌ Sample unmatched IDs: ['1729383171751970370', '1729621627626685788', '1729441997038520939', '1729858531511405156', '1729571847180094383']


---
## 🔍 ANALISIS SITUASI DATA

### Temuan:
| Sumber Data | Jumlah | Keterangan |
|-------------|--------|------------|
| **TiktokAdsCreativeData** | 2,702 product IDs | Data iklan (punya campaign name, video title) |
| **Excel Seller Center** | 111 produk | Referensi nama produk lengkap |
| **TiktokProduct table** | 92 produk | Data produk dari API TikTok |
| **Matched (Ads ↔ Excel)** | 93 produk | Product IDs yang sama |
| **Tidak ada di Excel** | 2,609 product IDs | Perlu dicari sumber namanya |

### Opsi Solusi:

| Opsi | Deskripsi | Pro | Kontra |
|------|-----------|-----|--------|
| **A. JOIN dengan Excel** | Merge Excel ke notebook | Simpel, tidak perlu upload | Hanya 93 produk dapat nama |
| **B. JOIN dengan TiktokProduct** | Merge dari table yang ada | Data sudah di DB | Hanya 92 produk |
| **C. Gunakan Campaign Name** | Pakai nama campaign | Sudah tersedia semua | Bukan nama produk spesifik |
| **D. Upload Excel ke DB** | Import Excel → TiktokProduct | Lebih rapi, reusable | Perlu script upload |
| **E. Kombinasi** | JOIN semua sumber | Coverage maksimal | Sedikit kompleks |

### 🎯 Rekomendasi: **OPSI E - Kombinasi**
Prioritas nama: `TiktokProduct.name` → `Excel.product_name` → `campaignName` → `videoTitle`

In [ ]:
# ============================================================
# CELL: Implementasi OPSI E - Gabungkan Semua Sumber Nama
# ============================================================
import pandas as pd
import sqlite3

# Setup paths
EXCEL_PATH = "../docs/ads_data/Tiktoksellercenter_batchedit_20260114_basic_information_template.xlsx"
DB_PATH = "../backend/config/databases/yumna_bertigamart.db"
TENANT_ID = "yumna_bertigamart"

# 1. Load semua data
print("🔄 Loading data dari semua sumber...")

# Excel reference
excel_df = pd.read_excel(EXCEL_PATH, skiprows=3)
excel_df.columns = ['product_id', 'sale_platforms', 'category', 'brand', 
                    'product_name', 'product_description', 'tts_url', 'toko_url']
excel_df['product_id'] = excel_df['product_id'].astype(str)
excel_lookup = excel_df.set_index('product_id')[['product_name', 'category']].to_dict('index')

# Database - TiktokProduct
conn = sqlite3.connect(DB_PATH)
tiktok_products = pd.read_sql_query("""
    SELECT productId, name FROM TiktokProduct WHERE tenantId = ?
""", conn, params=(TENANT_ID,))
tiktok_products['productId'] = tiktok_products['productId'].astype(str)
tiktok_lookup = tiktok_products.set_index('productId')['name'].to_dict()

# Database - Ads data (HANYA PERIODE VALID)
# Filter: periodStart harus dalam format ISO (mulai dengan '20')
ads_df = pd.read_sql_query("""
    SELECT * FROM TiktokAdsCreativeData 
    WHERE tenantId = ?
    AND periodStart LIKE '20%'
""", conn, params=(TENANT_ID,))

# Cek periode yang ada
periods_df = pd.read_sql_query("""
    SELECT 
        periodStart,
        periodEnd,
        COUNT(*) as records,
        COUNT(DISTINCT productId) as products
    FROM TiktokAdsCreativeData 
    WHERE tenantId = ?
    AND periodStart LIKE '20%'
    GROUP BY periodStart, periodEnd
    ORDER BY periodStart
""", conn, params=(TENANT_ID,))
conn.close()

print(f"   ✅ Excel: {len(excel_lookup)} produk")
print(f"   ✅ TiktokProduct: {len(tiktok_lookup)} produk")
print(f"   ✅ AdsCreativeData: {len(ads_df)} records")

# Tampilkan periode yang ada
print("\n" + "=" * 70)
print("📅 PERIODE DATA YANG VALID:")
print("=" * 70)
for _, row in periods_df.iterrows():
    start = str(row['periodStart'])[:10]
    end = str(row['periodEnd'])[:10]
    print(f"   {start} → {end}: {row['records']:,} records, {row['products']:,} produk")

print(f"\n   Total: {len(periods_df)} periode, {len(ads_df):,} records")

# 2. Fungsi untuk mendapatkan nama produk dengan prioritas
def get_product_name(row):
    """
    Prioritas nama:
    1. TiktokProduct.name (dari API)
    2. Excel product_name (dari Seller Center)
    3. campaignName (dari iklan)
    4. videoTitle (dari creative)
    5. Product ID (fallback)
    """
    product_id = str(row['productId'])
    
    # Priority 1: TiktokProduct table
    if product_id in tiktok_lookup:
        return tiktok_lookup[product_id]
    
    # Priority 2: Excel reference
    if product_id in excel_lookup:
        return excel_lookup[product_id]['product_name']
    
    # Priority 3: Campaign Name (jika informatif)
    campaign = row.get('campaignName', '')
    if campaign and pd.notna(campaign) and len(str(campaign)) > 10:
        # Skip jika hanya tanggal/angka
        if not str(campaign).startswith('Aktif sejak'):
            return campaign[:80]
    
    # Priority 4: Video Title
    video = row.get('videoTitle', '')
    if video and pd.notna(video) and len(str(video)) > 5:
        return str(video)[:80]
    
    # Fallback: Product ID (shortened)
    return f"Product-{product_id[-8:]}"

def get_product_category(product_id):
    """Get category from Excel if available."""
    pid = str(product_id)
    if pid in excel_lookup:
        return excel_lookup[pid].get('category', 'Tidak diketahui')
    return 'Tidak diketahui'

# 3. Apply nama ke dataframe
ads_df['product_name'] = ads_df.apply(get_product_name, axis=1)
ads_df['product_category'] = ads_df['productId'].apply(get_product_category)

# 4. Analisis coverage nama
def classify_name_source(row):
    pid = str(row['productId'])
    if pid in tiktok_lookup:
        return 'TiktokProduct'
    elif pid in excel_lookup:
        return 'Excel'
    elif pd.notna(row['campaignName']) and not str(row['campaignName']).startswith('Aktif'):
        return 'CampaignName'
    elif pd.notna(row['videoTitle']):
        return 'VideoTitle'
    else:
        return 'Fallback'

ads_df['name_source'] = ads_df.apply(classify_name_source, axis=1)

# 5. Summary
print("\n" + "=" * 70)
print("📊 HASIL MAPPING NAMA PRODUK")
print("=" * 70)
source_counts = ads_df.groupby('name_source')['productId'].nunique()
print("\nSumber nama berdasarkan unique products:")
for source, count in source_counts.items():
    print(f"   {source}: {count} produk")

print("\n✅ Sample hasil mapping:")
sample = ads_df[['productId', 'product_name', 'name_source', 'product_category']].drop_duplicates('productId').head(10)
display(sample)

🔄 Loading data dari semua sumber...
   ✅ Excel: 111 produk
   ✅ TiktokProduct: 0 produk
   ✅ AdsCreativeData: 83659 records

📊 HASIL MAPPING NAMA PRODUK

Sumber nama berdasarkan unique products:
   CampaignName: 646 produk
   Excel: 93 produk
   Fallback: 446 produk
   VideoTitle: 2080 produk

✅ Sample hasil mapping:


,productId,product_name,name_source,product_category
0,1729732509484222628,Nusseyba - Syakila French Khimar - Hijab Insta...,Excel,Khimar (601309)
1,1730228114064049316,Nusseyba - Sofa Inner - Ciput Hijab Anti Pusin...,Excel,Ciput (601305)
3,1730448467452069028,Nusseyba - Husna French Khimar - Jersey ITY FK...,Excel,Khimar (601309)
7,1730227817144878244,Nusseyba - Syakila French Khimar - Hijab Insta...,Excel,Khimar (601309)
8,1730483625184953508,Nusseyba - Husna French Khimar - Jersey ITY FK...,Excel,Khimar (601309)
9,1729583836961081508,Nusseyba - Zahiya Skirt - Rok Celana Jogger Sp...,Excel,Baju Olahraga Muslim (839048)
11,1731365288154334372,Nusseyba - Arabella Khimar Bandana - Jersey Kh...,Excel,Khimar (601309)
14,1730446350206600356,Nusseyba - Rubina Top - Atasan Dryfit - Outfit...,Excel,Kaos Olahraga (835720)
15,1729767299017376932,Nusseyba - Inayah Hat - Topi Haji dan Umroh -...,Excel,Topi (906248)
19,1731901363564741796,Nusseyba - Taza Skirt - Rok Celana Dry Fit - B...,Excel,Baju Olahraga Muslim (839048)


In [ ]:
# ============================================================
# CELL: Analisis Mendalam - Produk dengan Fallback Name
# ============================================================

# Lihat produk yang hanya punya fallback name (butuh nama)
fallback_products = ads_df[ads_df['name_source'] == 'Fallback']['productId'].unique()
print(f"🔍 Produk dengan Fallback name (butuh nama sebenarnya): {len(fallback_products)}")
print(f"\nSample product IDs yang perlu nama:")
for pid in fallback_products[:10]:
    print(f"   {pid}")

# Cek revenue dari produk tanpa nama vs dengan nama
print("\n" + "=" * 70)
print("💰 ANALISIS REVENUE BERDASARKAN SUMBER NAMA")
print("=" * 70)

revenue_by_source = ads_df.groupby('name_source').agg({
    'grossRevenue': 'sum',
    'cost': 'sum',
    'productId': 'nunique'
}).rename(columns={'productId': 'n_products'})

revenue_by_source['profit'] = revenue_by_source['grossRevenue'] - revenue_by_source['cost']
revenue_by_source['roi'] = revenue_by_source['grossRevenue'] / revenue_by_source['cost']
revenue_by_source['pct_revenue'] = revenue_by_source['grossRevenue'] / revenue_by_source['grossRevenue'].sum() * 100

display(revenue_by_source.sort_values('grossRevenue', ascending=False))

# Kesimpulan
print("\n" + "=" * 70)
print("🎯 KESIMPULAN")
print("=" * 70)
total_revenue = ads_df['grossRevenue'].sum()
fallback_revenue = ads_df[ads_df['name_source'] == 'Fallback']['grossRevenue'].sum()
pct_fallback = fallback_revenue / total_revenue * 100
print(f"""
Total Revenue: Rp {total_revenue:,.0f}
Revenue dari produk TANPA nama (Fallback): Rp {fallback_revenue:,.0f} ({pct_fallback:.1f}%)

Ini berarti {pct_fallback:.1f}% revenue berasal dari produk yang tidak punya nama referensi.
Untuk analisis yang lebih akurat, perlu mendapatkan nama produk ini.
""")

🔍 Produk dengan Fallback name (butuh nama sebenarnya): 446

Sample product IDs yang perlu nama:
   1732154033678157613
   -1
   1729608721857874103
   1732442739980928947
   1730334776250893781
   1729567830392080694
   1731183988721288828
   1731451212798723541
   1729648322317485288
   1732474800102868132

💰 ANALISIS REVENUE BERDASARKAN SUMBER NAMA


,grossRevenue,cost,n_products,profit,roi,pct_revenue
name_source,,,,,,
Excel,7.420882e+09,1.013317e+09,93,6.407565e+09,7.323355,99.808243
CampaignName,1.311298e+07,1.118956e+06,646,1.199403e+07,11.718946,0.176365
VideoTitle,7.631780e+05,2.280200e+04,2080,7.403760e+05,33.469783,0.010264
Fallback,3.812270e+05,1.935490e+05,446,1.876780e+05,1.969667,0.005127



🎯 KESIMPULAN

Total Revenue: Rp 7,435,139,610
Revenue dari produk TANPA nama (Fallback): Rp 381,227 (0.0%)

Ini berarti 0.0% revenue berasal dari produk yang tidak punya nama referensi.
Untuk analisis yang lebih akurat, perlu mendapatkan nama produk ini.



---
## 🎯 KESIMPULAN & REKOMENDASI MEKANISME

### Temuan Penting - Mapping Nama Produk:
| Sumber Nama | Produk | Revenue | % Revenue |
|-------------|--------|---------|-----------|
| **Excel** | 93 | Rp 7.4 Miliar | **99.8%** |
| CampaignName | 646 | Rp 13 Juta | 0.18% |
| VideoTitle | 2,080 | Rp 763K | 0.01% |
| Fallback (tanpa nama) | 446 | Rp 381K | 0.005% |

### 🔥 Temuan Penting - Jenis Materi Iklan:
| Jenis Materi | Creatives | Produk | Revenue | Avg ROI |
|--------------|-----------|--------|---------|---------|
| **Video** 🎬 | 83,315 | 2,699 | Rp 4.89 Miliar | 37.31 |
| **Kartu Produk** 🎴 | 344 | 92 | Rp 2.55 Miliar | 25.58 |

### ⚠️ PENTING: Analisis WAJIB Dipisahkan!
- **Video** dan **Kartu Produk** memiliki karakteristik berbeda
- ROI dan pola performa TIDAK BOLEH dicampur
- Kesimpulan harus **TERPISAH** untuk tiap jenis materi

### 🎯 Rekomendasi Mekanisme:

| Prioritas | Aksi | Alasan |
|-----------|------|--------|
| **1. TIDAK PERLU upload Excel ke DB** | Excel 111 produk sudah cover 99.8% revenue | Effort vs Impact tidak sebanding |
| **2. Gunakan JOIN di Notebook** | Merge Excel data langsung di analisis | Simpel, fleksibel |
| **3. PISAHKAN Analisis per creativeType** | Video ≠ Kartu Produk | Karakteristik berbeda |
| **4. 2 KESIMPULAN Terpisah** | Hasil ML untuk Video & Kartu Produk | Tidak dicampur |

### ✅ Arsitektur yang Direkomendasikan:

```
┌─────────────────────────────────────────────────────────────────┐
│                    NOTEBOOK ANALYSIS FLOW                        │
├─────────────────────────────────────────────────────────────────┤
│                                                                  │
│  [TiktokAdsCreativeData] ─┬─> FILTER creativeType='Video'       │
│        (Database)         │   └─> ML Analysis → Kesimpulan 🎬   │
│                           │                                      │
│  [Excel Seller Center] ───┼─> FILTER creativeType='Kartu produk'│
│        (File)             │   └─> ML Analysis → Kesimpulan 🎴   │
│                           │                                      │
│                           └─> JOIN → Enriched Data              │
└─────────────────────────────────────────────────────────────────┘
```

---

In [ ]:
# ============================================================
# CELL 2: CONFIGURATION
# ============================================================
import os

TENANT_ID = "yumna_bertigamart"
TOP_N = 10  # Number of products to show in each category

# Auto-detect database path (relatif dari folder notebooks)
DB_DIR = None
for path in ["../backend/config/databases"]:
    if os.path.exists(os.path.abspath(path)):
        DB_DIR = os.path.abspath(path)
        break

if DB_DIR is None:
    raise FileNotFoundError("Database directory not found! Make sure you're running from notebooks folder.")

DB_PATH = f"{DB_DIR}/{TENANT_ID}.db"

# Excel reference untuk nama produk (sekarang di folder ads_data dalam notebooks)
EXCEL_DIR = None
for path in ["ads_data"]:
    if os.path.exists(os.path.abspath(path)):
        EXCEL_DIR = os.path.abspath(path)
        break

if EXCEL_DIR is None:
    raise FileNotFoundError("ads_data directory not found! Make sure you're running from notebooks folder.")

EXCEL_PATH = f"{EXCEL_DIR}/Tiktoksellercenter_batchedit_20260114_basic_information_template.xlsx"

print(f"📋 Configuration:")
print(f"   Tenant: {TENANT_ID}")
print(f"   Database: {DB_PATH}")
print(f"   Excel Reference: {EXCEL_PATH}")
print(f"   Top N Display: {TOP_N}")

📋 Configuration:
   Tenant: yumna_bertigamart
   Database: c:\Users\yumna\Desktop\Project\omni\backend\config\databases/yumna_bertigamart.db
   Excel Reference: c:\Users\yumna\Desktop\Project\omni\docs\ads_data/Tiktoksellercenter_batchedit_20260114_basic_information_template.xlsx
   Top N Display: 10


In [ ]:
# ============================================================
# CELL 3: IMPORTS & UTILITIES
# ============================================================
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from scipy import stats
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# Visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print("✅ Libraries loaded")
print(f"   scipy.stats: Mann-Kendall, Linear Regression")
print(f"   numpy: Z-score, statistical functions")
print(f"   plotly: Interactive visualizations")

✅ Libraries loaded
   scipy.stats: Mann-Kendall, Linear Regression
   numpy: Z-score, statistical functions
   plotly: Interactive visualizations


In [ ]:
# ============================================================
# CELL 4: LOAD DATA & ENRICH WITH PRODUCT NAMES + SPLIT BY CREATIVE TYPE
# ============================================================
conn = sqlite3.connect(DB_PATH)

# FILTER HANYA PERIODE VALID (format ISO, bukan timestamp corrupt)
df = pd.read_sql_query("""
    SELECT * FROM TiktokAdsCreativeData 
    WHERE tenantId = ?
    AND periodStart LIKE '20%'
""", conn, params=(TENANT_ID,))

# Cek info periode yang tersedia
periods_info = pd.read_sql_query("""
    SELECT 
        periodStart,
        periodEnd,
        COUNT(*) as records,
        COUNT(DISTINCT productId) as products,
        SUM(cost) as total_cost,
        SUM(grossRevenue) as total_revenue
    FROM TiktokAdsCreativeData 
    WHERE tenantId = ?
    AND periodStart LIKE '20%'
    GROUP BY periodStart, periodEnd
    ORDER BY periodStart
""", conn, params=(TENANT_ID,))

conn.close()

# Parse dates
df['periodStart'] = pd.to_datetime(df['periodStart'], errors='coerce')
df['periodEnd'] = pd.to_datetime(df['periodEnd'], errors='coerce')

print("=" * 70)
print("📅 PERIODE DATA YANG DIANALISIS (HANYA DATA VALID)")
print("=" * 70)
n_periods = len(periods_info)
for idx, row in periods_info.iterrows():
    start = str(row['periodStart'])[:10]
    end = str(row['periodEnd'])[:10]
    roi = row['total_revenue'] / row['total_cost'] if row['total_cost'] > 0 else 0
    print(f"  {idx+1}. {start} → {end}: {row['records']:,} records, {row['products']:,} produk, ROI {roi:.2f}x")

print(f"\n  Total: {n_periods} periode, {len(df):,} records")
print(f"  💡 Mann-Kendall Test membutuhkan min. 4 periode untuk valid.")
print(f"  {'✅ Data cukup untuk trend analysis!' if n_periods >= 4 else f'⚠️ Perlu {4-n_periods} periode lagi untuk analisis trend optimal.'}")

# ============================================================
# ENRICHMENT: Tambahkan nama produk dari Excel reference
# ============================================================
print("\n🔄 Enriching data dengan nama produk dari Excel...")

# Load Excel reference
excel_ref = pd.read_excel(EXCEL_PATH, skiprows=3)
excel_ref.columns = ['product_id', 'sale_platforms', 'category', 'brand', 
                     'product_name', 'product_description', 'tts_url', 'toko_url']
excel_ref['product_id'] = excel_ref['product_id'].astype(str)
excel_lookup = excel_ref.set_index('product_id')[['product_name', 'category']].to_dict('index')

def get_product_name(row):
    """Prioritas nama: Excel → campaignName → videoTitle → fallback"""
    product_id = str(row['productId'])
    if product_id in excel_lookup:
        return excel_lookup[product_id]['product_name']
    campaign = row.get('campaignName', '')
    if campaign and pd.notna(campaign) and len(str(campaign)) > 10:
        if not str(campaign).startswith('Aktif sejak'):
            return str(campaign)[:80]
    video = row.get('videoTitle', '')
    if video and pd.notna(video) and len(str(video)) > 5:
        return str(video)[:80]
    return f"Product-{product_id[-8:]}"

def get_product_category(product_id):
    """Get category from Excel if available."""
    pid = str(product_id)
    if pid in excel_lookup:
        cat = excel_lookup[pid].get('category', None)
        return cat if pd.notna(cat) else 'Tidak dikategorikan'
    return 'Tidak dikategorikan'

# Apply enrichment
df['product_name'] = df.apply(get_product_name, axis=1)
df['product_category'] = df['productId'].apply(get_product_category)
df['_name_from_excel'] = df['productId'].astype(str).isin(excel_lookup.keys())

# ============================================================
# SPLIT DATA BY CREATIVE TYPE (PENTING!)
# ============================================================
print("\n" + "=" * 70)
print("🎯 MEMISAHKAN DATA BERDASARKAN JENIS MATERI IKLAN")
print("=" * 70)

# Definisi creative types
CREATIVE_TYPES = ['Video', 'Kartu produk']

# Create dictionary to store split dataframes
df_by_type = {}
for ct in CREATIVE_TYPES:
    df_by_type[ct] = df[df['creativeType'] == ct].copy()
    icon = "🎬" if ct == "Video" else "🎴"
    subset = df_by_type[ct]
    print(f"\n{icon} {ct.upper()}:")
    print(f"   Records: {len(subset):,}")
    print(f"   Unique Products: {subset['productId'].nunique()}")
    print(f"   Total Revenue: Rp {subset['grossRevenue'].sum():,.0f}")
    print(f"   Total Cost: Rp {subset['cost'].sum():,.0f}")
    print(f"   Avg ROI: {subset['roi'].mean():.2f}")

# Store in global variables for use in subsequent cells
df_video = df_by_type.get('Video', pd.DataFrame())
df_kartu = df_by_type.get('Kartu produk', pd.DataFrame())

📅 PERIODE DATA YANG DIANALISIS (HANYA DATA VALID)
  1. 2025-11-01 → 2025-11-30: 22,027 records, 2,692 produk, ROI 6.98x
  2. 2025-12-01 → 2025-12-31: 21,953 records, 2,690 produk, ROI 7.92x
  3. 2026-01-01 → 2026-01-12: 21,937 records, 2,679 produk, ROI 7.62x

  Total: 3 periode, 65,917 records
  💡 Mann-Kendall Test membutuhkan min. 4 periode untuk valid.
  ⚠️ Perlu 1 periode lagi untuk analisis trend optimal.

🔄 Enriching data dengan nama produk dari Excel...

🎯 MEMISAHKAN DATA BERDASARKAN JENIS MATERI IKLAN

🎬 VIDEO:
   Records: 65,714
   Unique Products: 2691
   Total Revenue: Rp 1,390,772,327
   Total Cost: Rp 261,301,367
   Avg ROI: 21.01

🎴 KARTU PRODUK:
   Records: 203
   Unique Products: 73
   Total Revenue: Rp 754,173,548
   Total Cost: Rp 26,851,781
   Avg ROI: 28.34


In [ ]:
# ============================================================
# CELL 5: STATISTICAL FUNCTIONS
# ============================================================

def mann_kendall_test(data: np.ndarray) -> Dict:
    """
    Mann-Kendall Trend Test - Non-parametric test for monotonic trend.
    Used to detect if there's a statistically significant trend.
    
    Returns:
        trend: 'increasing', 'decreasing', or 'no_trend'
        p_value: statistical significance
        tau: Kendall's tau correlation
    """
    n = len(data)
    if n < 4:
        return {'trend': 'insufficient_data', 'p_value': 1.0, 'tau': 0}
    
    # Calculate S statistic
    s = 0
    for i in range(n-1):
        for j in range(i+1, n):
            s += np.sign(data[j] - data[i])
    
    # Variance
    var_s = (n * (n-1) * (2*n+5)) / 18
    
    # Z statistic
    if s > 0:
        z = (s - 1) / np.sqrt(var_s)
    elif s < 0:
        z = (s + 1) / np.sqrt(var_s)
    else:
        z = 0
    
    # P-value (two-tailed)
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))
    
    # Kendall's tau
    tau = s / (n * (n-1) / 2)
    
    # Determine trend
    if p_value < 0.05:
        trend = 'increasing' if s > 0 else 'decreasing'
    else:
        trend = 'no_trend'
    
    return {'trend': trend, 'p_value': p_value, 'tau': tau, 'z': z}


def linear_trend_analysis(data: np.ndarray) -> Dict:
    """
    Linear Regression Trend Analysis.
    Fits a line to the data to determine slope and R².
    
    Returns:
        slope: rate of change per period
        r_squared: how well the line fits
        direction: 'up', 'down', or 'flat'
    """
    n = len(data)
    if n < 2:
        return {'slope': 0, 'r_squared': 0, 'direction': 'insufficient_data'}
    
    x = np.arange(n)
    slope, intercept, r_value, p_value, std_err = stats.linregress(x, data)
    
    # Determine direction based on slope significance
    if abs(slope) < 0.01 * np.mean(data) or p_value > 0.1:
        direction = 'flat'
    elif slope > 0:
        direction = 'up'
    else:
        direction = 'down'
    
    return {
        'slope': slope,
        'r_squared': r_value ** 2,
        'p_value': p_value,
        'direction': direction
    }


def calculate_momentum(recent: float, historical: float) -> Tuple[float, str]:
    """
    Calculate momentum by comparing recent vs historical performance.
    
    Returns:
        momentum_score: percentage change
        momentum_label: 'improving', 'declining', 'stable'
    """
    if historical == 0:
        return (0, 'new')
    
    change = (recent - historical) / abs(historical) * 100
    
    if change > 20:
        return (change, 'strongly_improving')
    elif change > 5:
        return (change, 'improving')
    elif change < -20:
        return (change, 'strongly_declining')
    elif change < -5:
        return (change, 'declining')
    else:
        return (change, 'stable')


def coefficient_of_variation(data: np.ndarray) -> float:
    """
    Calculate Coefficient of Variation (CV).
    Measures consistency - lower CV = more consistent performance.
    """
    mean = np.mean(data)
    if mean == 0:
        return float('inf')
    return (np.std(data) / abs(mean)) * 100


print("✅ Statistical functions loaded:")
print("   - Mann-Kendall Trend Test")
print("   - Linear Regression Analysis")
print("   - Momentum Calculator")
print("   - Coefficient of Variation")

✅ Statistical functions loaded:
   - Mann-Kendall Trend Test
   - Linear Regression Analysis
   - Momentum Calculator
   - Coefficient of Variation


In [ ]:
# ============================================================
00# CELL 6: ANALISIS PRODUK - DIPISAHKAN PER JENIS MATERI IKLAN
# ============================================================

def get_best_name(series):
    """Ambil nama yang paling sering muncul (bukan kosong)."""
    valid = series.dropna()
    if len(valid) > 0:
        valid = valid[valid.astype(str).str.strip() != '']
    if len(valid) > 0:
        mode_vals = valid.mode()
        return mode_vals.iloc[0] if len(mode_vals) > 0 else valid.iloc[0]
    return None

def analyze_product_fast(product_df: pd.DataFrame) -> Optional[Dict]:
    """Analisis produk secara cepat dengan metode statistik."""
    try:
        if len(product_df) == 0:
            return None
        product_id = product_df['productId'].iloc[0]
        campaign_name = get_best_name(product_df['campaignName'])
        video_title = get_best_name(product_df['videoTitle'])
        
        total_cost = product_df['cost'].sum()
        if total_cost <= 0:
            return None
        total_revenue = product_df['grossRevenue'].sum()
        total_orders = product_df['ordersSku'].sum()
        profit = total_revenue - total_cost
        roi = total_revenue / total_cost
        
        product_df = product_df.sort_values('periodStart')
        n_periods = len(product_df)
        roi_series = product_df['roi'].values
        
        # Mann-Kendall test
        if n_periods >= 4:
            roi_trend = mann_kendall_test(roi_series)
            roi_trend_label = roi_trend['trend']
            roi_trend_p = roi_trend['p_value']
        else:
            roi_trend_label, roi_trend_p = 'data_kurang', 1.0
        
        # Linear regression
        if n_periods >= 2:
            x = np.arange(n_periods)
            slope, _, r_value, p_value, _ = stats.linregress(x, roi_series)
            roi_slope = slope
            if abs(slope) < 0.01 * np.mean(roi_series) or p_value > 0.1:
                roi_direction = 'stabil'
            elif slope > 0:
                roi_direction = 'naik'
            else:
                roi_direction = 'turun'
        else:
            roi_slope, roi_direction = 0, 'data_kurang'
        
        # Momentum
        if n_periods >= 3:
            split_idx = max(1, int(n_periods * 0.7))
            historical_roi = roi_series[:split_idx].mean()
            recent_roi = roi_series[split_idx:].mean()
            momentum_score = ((recent_roi - historical_roi) / abs(historical_roi) * 100) if historical_roi != 0 else 0
            if momentum_score > 20: momentum_label = 'meningkat_pesat'
            elif momentum_score > 5: momentum_label = 'meningkat'
            elif momentum_score < -20: momentum_label = 'menurun_drastis'
            elif momentum_score < -5: momentum_label = 'menurun'
            else: momentum_label = 'stabil'
        else:
            momentum_score, momentum_label = 0, 'data_kurang'
            recent_roi, historical_roi = roi, roi
        
        roi_mean = np.mean(roi_series)
        roi_cv = (np.std(roi_series) / abs(roi_mean) * 100) if roi_mean != 0 else float('inf')
        
        return {
            'productId': product_id, 'campaignName': campaign_name, 'videoTitle': video_title,
            'n_periods': n_periods, 'total_cost': total_cost, 'total_revenue': total_revenue,
            'total_orders': total_orders, 'profit': profit, 'roi': roi,
            'roi_trend': roi_trend_label, 'roi_trend_p': roi_trend_p, 'roi_direction': roi_direction,
            'roi_slope': roi_slope, 'momentum_score': momentum_score, 'momentum_label': momentum_label,
            'roi_cv': roi_cv, 'avg_roi': roi_mean, 'recent_roi': recent_roi, 'historical_roi': historical_roi
        }
    except Exception as e:
        return None

def analyze_creative_type_batch(source_df: pd.DataFrame, creative_type: str, top_n: int = 200) -> pd.DataFrame:
    """Analisis batch untuk satu jenis materi iklan."""
    icon = "🎬" if creative_type == "Video" else "🎴"
    print(f"\n{icon} Menganalisis {creative_type.upper()}...")
    
    if len(source_df) == 0:
        print(f"   ⚠️ Tidak ada data untuk {creative_type}")
        return pd.DataFrame()
    
    # Step 1: Agregasi dasar
    basic_agg = source_df.groupby('productId').agg({
        'cost': 'sum', 'grossRevenue': 'sum', 'ordersSku': 'sum',
        'roi': ['mean', 'std', 'count'],
        'campaignName': get_best_name, 'videoTitle': get_best_name,
        'product_name': 'first', 'product_category': 'first',
        'periodStart': 'min', 'periodEnd': 'max'
    }).reset_index()
    basic_agg.columns = ['productId', 'total_cost', 'total_revenue', 'total_orders', 
                         'avg_roi', 'roi_std', 'n_periods', 'campaignName', 'videoTitle',
                         'product_name', 'product_category', 'first_date', 'last_date']
    
    basic_agg = basic_agg[basic_agg['total_cost'] > 0]
    print(f"   Ditemukan {len(basic_agg)} produk dengan pengeluaran")
    
    # Metrik turunan
    basic_agg['profit'] = basic_agg['total_revenue'] - basic_agg['total_cost']
    basic_agg['roi'] = basic_agg['total_revenue'] / basic_agg['total_cost']
    basic_agg['roi_cv'] = (basic_agg['roi_std'] / basic_agg['avg_roi'].abs() * 100).fillna(0)
    basic_agg['creative_type'] = creative_type
    
    # Step 2: Analisis trend untuk produk teratas
    top_products = basic_agg.nlargest(top_n, 'total_revenue')['productId'].tolist()
    trend_results = []
    for product_id in top_products:
        product_df = source_df[source_df['productId'] == product_id]
        result = analyze_product_fast(product_df)
        if result:
            trend_results.append({
                'productId': product_id, 'roi_trend': result['roi_trend'],
                'roi_direction': result['roi_direction'], 'roi_slope': result['roi_slope'],
                'momentum_score': result['momentum_score'], 'momentum_label': result['momentum_label'],
                'recent_roi': result['recent_roi'], 'historical_roi': result['historical_roi']
            })
    
    # Gabungkan
    trend_df = pd.DataFrame(trend_results)
    analysis_df = basic_agg.merge(trend_df, on='productId', how='left')
    
    # Fill NA
    analysis_df['roi_trend'] = analysis_df['roi_trend'].fillna('belum_dianalisis')
    analysis_df['roi_direction'] = analysis_df['roi_direction'].fillna('belum_dianalisis')
    analysis_df['momentum_label'] = analysis_df['momentum_label'].fillna('belum_dianalisis')
    analysis_df['momentum_score'] = analysis_df['momentum_score'].fillna(0)
    analysis_df['recent_roi'] = analysis_df['recent_roi'].fillna(analysis_df['avg_roi'])
    analysis_df['historical_roi'] = analysis_df['historical_roi'].fillna(analysis_df['avg_roi'])
    
    print(f"   ✅ Selesai: {len(analysis_df)} produk, {len(trend_results)} dengan trend lengkap")
    return analysis_df

# ============================================================
# JALANKAN ANALISIS TERPISAH
# ============================================================
print("=" * 70)
print("🔄 ANALISIS PRODUK - DIPISAHKAN PER JENIS MATERI IKLAN")
print("=" * 70)
start_time = datetime.now()

# Analisis terpisah untuk Video dan Kartu Produk
analysis_video = analyze_creative_type_batch(df_video, 'Video', top_n=200)
analysis_kartu = analyze_creative_type_batch(df_kartu, 'Kartu produk', top_n=100)

elapsed = (datetime.now() - start_time).total_seconds()
print(f"\n✅ Total waktu analisis: {elapsed:.1f} detik")

# Summary
print("\n" + "=" * 70)
print("📊 RINGKASAN HASIL ANALISIS")
print("=" * 70)
print(f"🎬 VIDEO: {len(analysis_video)} produk dianalisis")
print(f"🎴 KARTU PRODUK: {len(analysis_kartu)} produk dianalisis")

🔄 ANALISIS PRODUK - DIPISAHKAN PER JENIS MATERI IKLAN

🎬 Menganalisis VIDEO...
   Ditemukan 87 produk dengan pengeluaran
   ✅ Selesai: 87 produk, 87 dengan trend lengkap

🎴 Menganalisis KARTU PRODUK...
   Ditemukan 92 produk dengan pengeluaran
   ✅ Selesai: 92 produk, 92 dengan trend lengkap

✅ Total waktu analisis: 258.6 detik

📊 RINGKASAN HASIL ANALISIS
🎬 VIDEO: 87 produk dianalisis
🎴 KARTU PRODUK: 92 produk dianalisis


In [ ]:
# ============================================================
# CELL 7: SISTEM SCORING KOMPOSIT - UNTUK KEDUA JENIS MATERI
# ============================================================

def calculate_composite_score(row: pd.Series) -> float:
    """Menghitung skor komposit dengan pembobotan."""
    score = 0
    roi_score = min(row['roi'] / 10 * 100, 100)
    score += roi_score * 0.30
    
    if row['profit'] > 0:
        profit_score = min(row['profit'] / 100000000 * 100, 100)
    else:
        profit_score = max(row['profit'] / 10000000 * 10, -50)
    score += profit_score * 0.25
    
    momentum_map = {
        'meningkat_pesat': 100, 'strongly_improving': 100,
        'meningkat': 75, 'improving': 75,
        'stabil': 50, 'stable': 50,
        'menurun': 25, 'declining': 25,
        'menurun_drastis': 0, 'strongly_declining': 0,
        'data_kurang': 50, 'insufficient_data': 50,
        'belum_dianalisis': 50, 'not_analyzed': 50
    }
    momentum_score = momentum_map.get(row['momentum_label'], 50)
    score += momentum_score * 0.20
    
    if row['roi_cv'] != float('inf') and not pd.isna(row['roi_cv']):
        consistency_score = max(100 - row['roi_cv'], 0)
    else:
        consistency_score = 50
    score += consistency_score * 0.15
    
    trend_map = {
        'increasing': 100, 'naik': 100, 'no_trend': 50, 'stabil': 50,
        'decreasing': 0, 'turun': 0, 'data_kurang': 50, 'belum_dianalisis': 50
    }
    trend_score = trend_map.get(row['roi_trend'], 50)
    score += trend_score * 0.10
    
    return score

def categorize_product(row: pd.Series) -> str:
    """Kategorisasi produk berdasarkan analisis multi-faktor."""
    momentum = row['momentum_label']
    improving = momentum in ['meningkat', 'meningkat_pesat', 'improving', 'strongly_improving']
    declining = momentum in ['menurun', 'menurun_drastis', 'declining', 'strongly_declining']
    
    if row['profit'] < 0 and declining: return 'HENTIKAN'
    if row['roi'] < 0.5 and not improving: return 'HENTIKAN'
    if row['profit'] < 0 and improving: return 'PANTAU'
    if row['roi'] < 2 and improving: return 'PANTAU'
    if row['roi'] >= 2 and not declining: return 'LANJUTKAN'
    if row['roi'] >= 2 and declining: return 'PANTAU'
    if row['composite_score'] >= 60: return 'LANJUTKAN'
    elif row['composite_score'] >= 40: return 'PANTAU'
    else: return 'HENTIKAN'

# ============================================================
# APPLY SCORING KE KEDUA JENIS MATERI IKLAN
# ============================================================
print("=" * 70)
print("🎯 SCORING & KATEGORISASI - PER JENIS MATERI IKLAN")
print("=" * 70)

# Process Video
if len(analysis_video) > 0:
    analysis_video['composite_score'] = analysis_video.apply(calculate_composite_score, axis=1)
    analysis_video['category'] = analysis_video.apply(categorize_product, axis=1)
    
    print("\n🎬 VIDEO ADS:")
    for cat in ['LANJUTKAN', 'PANTAU', 'HENTIKAN']:
        count = len(analysis_video[analysis_video['category'] == cat])
        icon = '🟢' if cat == 'LANJUTKAN' else '🟡' if cat == 'PANTAU' else '🔴'
        print(f"   {icon} {cat}: {count} produk")

# Process Kartu Produk
if len(analysis_kartu) > 0:
    analysis_kartu['composite_score'] = analysis_kartu.apply(calculate_composite_score, axis=1)
    analysis_kartu['category'] = analysis_kartu.apply(categorize_product, axis=1)
    
    print("\n🎴 KARTU PRODUK:")
    for cat in ['LANJUTKAN', 'PANTAU', 'HENTIKAN']:
        count = len(analysis_kartu[analysis_kartu['category'] == cat])
        icon = '🟢' if cat == 'LANJUTKAN' else '🟡' if cat == 'PANTAU' else '🔴'
        print(f"   {icon} {cat}: {count} produk")

# Gabungkan untuk referensi (tapi KESIMPULAN tetap terpisah!)
analysis_all = pd.concat([analysis_video, analysis_kartu], ignore_index=True)
print(f"\n📊 Total semua produk: {len(analysis_all)}")
print("⚠️ CATATAN: Kesimpulan dan rekomendasi akan TERPISAH per jenis materi iklan!")

🎯 SCORING & KATEGORISASI - PER JENIS MATERI IKLAN

🎬 VIDEO ADS:
   🟢 LANJUTKAN: 27 produk
   🟡 PANTAU: 27 produk
   🔴 HENTIKAN: 33 produk

🎴 KARTU PRODUK:
   🟢 LANJUTKAN: 50 produk
   🟡 PANTAU: 33 produk
   🔴 HENTIKAN: 9 produk

📊 Total semua produk: 179
⚠️ CATATAN: Kesimpulan dan rekomendasi akan TERPISAH per jenis materi iklan!


In [ ]:
# ============================================================
# CELL 8: GENERATE REKOMENDASI - UNTUK KEDUA JENIS MATERI
# ============================================================

def generate_recommendation_text(row: pd.Series) -> str:
    """Membuat teks rekomendasi detail dengan bukti pendukung."""
    reasons = []
    
    if row['roi'] >= 5: reasons.append(f"✅ ROI Sangat Baik: {row['roi']:.2f}x")
    elif row['roi'] >= 3: reasons.append(f"✅ ROI Baik: {row['roi']:.2f}x")
    elif row['roi'] >= 1: reasons.append(f"⚠️ ROI Rendah: {row['roi']:.2f}x")
    else: reasons.append(f"❌ ROI Negatif: {row['roi']:.2f}x")
    
    if row['profit'] > 0: reasons.append(f"✅ Profit: Rp {row['profit']:,.0f}")
    else: reasons.append(f"❌ Kerugian: Rp {abs(row['profit']):,.0f}")
    
    momentum = row['momentum_label']
    mscore = row['momentum_score']
    if momentum in ['meningkat_pesat', 'strongly_improving']: reasons.append(f"📈 Momentum Kuat: +{mscore:.1f}%")
    elif momentum in ['meningkat', 'improving']: reasons.append(f"📈 Membaik: +{mscore:.1f}%")
    elif momentum in ['menurun_drastis', 'strongly_declining']: reasons.append(f"📉 Penurunan Tajam: {mscore:.1f}%")
    elif momentum in ['menurun', 'declining']: reasons.append(f"📉 Menurun: {mscore:.1f}%")
    else: reasons.append(f"➡️ Stabil")
    
    roi_trend = row.get('roi_trend', 'belum_dianalisis')
    if roi_trend in ['increasing', 'naik']: reasons.append(f"📊 Trend Naik")
    elif roi_trend in ['decreasing', 'turun']: reasons.append(f"📊 Trend Turun")
    
    reasons.append(f"📅 {row['n_periods']} periode, {row['total_orders']:,.0f} pesanan")
    return " | ".join(reasons)

def get_display_name(row):
    """Get best display name for product."""
    if 'product_name' in row and pd.notna(row.get('product_name')) and str(row.get('product_name', '')).strip():
        return str(row['product_name'])[:60]
    if pd.notna(row.get('campaignName')) and str(row.get('campaignName', '')).strip():
        name = str(row['campaignName'])
        if not name.startswith('Aktif sejak'):
            return name[:60]
    if pd.notna(row.get('videoTitle')) and str(row.get('videoTitle', '')).strip():
        return str(row['videoTitle'])[:60]
    return f"Product-{str(row['productId'])[-8:]}"

# Apply rekomendasi ke kedua dataset
if len(analysis_video) > 0:
    analysis_video['rekomendasi'] = analysis_video.apply(generate_recommendation_text, axis=1)
    analysis_video['display_name'] = analysis_video.apply(get_display_name, axis=1)
    
if len(analysis_kartu) > 0:
    analysis_kartu['rekomendasi'] = analysis_kartu.apply(generate_recommendation_text, axis=1)
    analysis_kartu['display_name'] = analysis_kartu.apply(get_display_name, axis=1)

# Update analysis_all juga
analysis_all = pd.concat([analysis_video, analysis_kartu], ignore_index=True)

print("✅ Rekomendasi dengan bukti data telah dibuat untuk kedua jenis materi iklan")

✅ Rekomendasi dengan bukti data telah dibuat untuk kedua jenis materi iklan


---
## 🟢 PRODUK LAYAK DILANJUTKAN (TERPISAH PER JENIS MATERI)
Produk dengan performa kuat - lanjutkan atau tingkatkan budget iklan
### ⚠️ Hasil dipisahkan: 🎬 VIDEO dan 🎴 KARTU PRODUK

In [ ]:
# ============================================================
# CELL 9: PRODUK LAYAK DILANJUTKAN - TERPISAH PER JENIS MATERI
# ============================================================

def display_products_by_category(df_source, creative_type, category, top_n=10, color_scale='Greens'):
    """Display products for a specific category and creative type."""
    icon = "🎬" if creative_type == "Video" else "🎴"
    cat_icon = '🟢' if category == 'LANJUTKAN' else '🟡' if category == 'PANTAU' else '🔴'
    
    products = df_source[df_source['category'] == category].sort_values(
        'composite_score', ascending=(category == 'HENTIKAN')
    ).head(top_n)
    
    if len(products) == 0:
        print(f"\n{icon} {creative_type.upper()} - {cat_icon} {category}: Tidak ada produk")
        return None
    
    print(f"\n{'='*80}")
    print(f"{icon} {creative_type.upper()} - {cat_icon} TOP {min(top_n, len(products))} {category}")
    print(f"{'='*80}")
    
    for i, (_, row) in enumerate(products.iterrows(), 1):
        print(f"\n{i}. 📦 {row['display_name']}")
        print(f"   ID: {row['productId']}")
        print(f"   🎯 Skor: {row['composite_score']:.1f}/100 | ROI: {row['roi']:.2f}x | Profit: Rp {row['profit']:,.0f}")
        print(f"   📊 {row['rekomendasi']}")
    
    return products

print("=" * 80)
print("🟢 PRODUK LAYAK DILANJUTKAN - TERPISAH PER JENIS MATERI IKLAN")
print("=" * 80)
print("Produk-produk ini memiliki performa baik. Lanjutkan atau tingkatkan budget.\n")

# Video - LANJUTKAN
keep_video = display_products_by_category(analysis_video, 'Video', 'LANJUTKAN', TOP_N)

# Kartu Produk - LANJUTKAN  
keep_kartu = display_products_by_category(analysis_kartu, 'Kartu produk', 'LANJUTKAN', TOP_N)

# Visualisasi terpisah
if keep_video is not None and len(keep_video) > 0:
    fig1 = px.bar(keep_video, x='display_name', y='roi', color='composite_score',
                  title='🎬 VIDEO - Produk Layak Dilanjutkan',
                  color_continuous_scale='Greens')
    fig1.add_hline(y=3, line_dash="dash", line_color="green", annotation_text="Target ROI")
    fig1.update_layout(xaxis_title='Produk', yaxis_title='ROI', xaxis_tickangle=-45)
    fig1.show()

if keep_kartu is not None and len(keep_kartu) > 0:
    fig2 = px.bar(keep_kartu, x='display_name', y='roi', color='composite_score',
                  title='🎴 KARTU PRODUK - Produk Layak Dilanjutkan',
                  color_continuous_scale='Greens')
    fig2.add_hline(y=3, line_dash="dash", line_color="green", annotation_text="Target ROI")
    fig2.update_layout(xaxis_title='Produk', yaxis_title='ROI', xaxis_tickangle=-45)
    fig2.show()

🟢 PRODUK LAYAK DILANJUTKAN - TERPISAH PER JENIS MATERI IKLAN
Produk-produk ini memiliki performa baik. Lanjutkan atau tingkatkan budget.


🎬 VIDEO - 🟢 TOP 10 LANJUTKAN

1. 📦 Nusseyba - Ishana Bucket Hat - Topi Umrah Wanita - Topi Haji
   ID: 1729511392877709476
   🎯 Skor: 78.2/100 | ROI: 10.42x | Profit: Rp 92,696,981
   📊 ✅ ROI Sangat Baik: 10.42x | ✅ Profit: Rp 92,696,981 | 📈 Momentum Kuat: +111.6% | 📅 3644 periode, 578 pesanan

2. 📦 Nusseyba - Sofa Inner - Ciput Hijab Anti Pusing- Dalaman Ker
   ID: 1730228114064049316
   🎯 Skor: 74.9/100 | ROI: 6.64x | Profit: Rp 913,281,023
   📊 ✅ ROI Sangat Baik: 6.64x | ✅ Profit: Rp 913,281,023 | 📈 Momentum Kuat: +946.3% | 📊 Trend Naik | 📅 2406 periode, 24,110 pesanan

3. 📦 Nusseyba - Basma Cap - Topi Baseball Cap Unisex - Topi Baseb
   ID: 1730228438133802148
   🎯 Skor: 72.9/100 | ROI: 5.97x | Profit: Rp 102,683,166
   📊 ✅ ROI Sangat Baik: 5.97x | ✅ Profit: Rp 102,683,166 | 📈 Momentum Kuat: +84.3% | 📊 Trend Naik | 📅 922 periode, 852 pesanan

4.

---
## 🔴 PRODUK SEBAIKNYA DIHENTIKAN (TERPISAH PER JENIS MATERI)
Produk merugi tanpa tanda perbaikan - hentikan segera untuk menghemat budget
### ⚠️ Hasil dipisahkan: 🎬 VIDEO dan 🎴 KARTU PRODUK

In [ ]:
# ============================================================
# CELL 10: PRODUK SEBAIKNYA DIHENTIKAN - TERPISAH PER JENIS MATERI
# ============================================================

print("=" * 80)
print("🔴 PRODUK SEBAIKNYA DIHENTIKAN - TERPISAH PER JENIS MATERI IKLAN")
print("=" * 80)
print("Produk-produk ini merugi. Hentikan iklan segera untuk menghemat budget.\n")

# Hitung potensi penghematan
loss_video = analysis_video[(analysis_video['category'] == 'HENTIKAN') & (analysis_video['profit'] < 0)]['profit'].sum()
loss_kartu = analysis_kartu[(analysis_kartu['category'] == 'HENTIKAN') & (analysis_kartu['profit'] < 0)]['profit'].sum()

print(f"💸 POTENSI PENGHEMATAN TOTAL:")
print(f"   🎬 Video: Rp {abs(loss_video):,.0f}")
print(f"   🎴 Kartu Produk: Rp {abs(loss_kartu):,.0f}")
print(f"   📊 TOTAL: Rp {abs(loss_video + loss_kartu):,.0f}")

# Video - HENTIKAN
pause_video = display_products_by_category(analysis_video, 'Video', 'HENTIKAN', TOP_N)

# Kartu Produk - HENTIKAN
pause_kartu = display_products_by_category(analysis_kartu, 'Kartu produk', 'HENTIKAN', TOP_N)

# Visualisasi terpisah
if pause_video is not None and len(pause_video) > 0:
    fig1 = px.bar(pause_video, x='display_name', y='profit', color='roi',
                  title='🎬 VIDEO - Produk untuk Dihentikan',
                  color_continuous_scale='RdYlGn')
    fig1.add_hline(y=0, line_dash="dash", line_color="red", annotation_text="Break-even")
    fig1.update_layout(xaxis_title='Produk', yaxis_title='Profit (Rp)', xaxis_tickangle=-45)
    fig1.show()

if pause_kartu is not None and len(pause_kartu) > 0:
    fig2 = px.bar(pause_kartu, x='display_name', y='profit', color='roi',
                  title='🎴 KARTU PRODUK - Produk untuk Dihentikan',
                  color_continuous_scale='RdYlGn')
    fig2.add_hline(y=0, line_dash="dash", line_color="red", annotation_text="Break-even")
    fig2.update_layout(xaxis_title='Produk', yaxis_title='Profit (Rp)', xaxis_tickangle=-45)
    fig2.show()

🔴 PRODUK SEBAIKNYA DIHENTIKAN - TERPISAH PER JENIS MATERI IKLAN
Produk-produk ini merugi. Hentikan iklan segera untuk menghemat budget.

💸 POTENSI PENGHEMATAN TOTAL:
   🎬 Video: Rp 1,069,653
   🎴 Kartu Produk: Rp 783,929
   📊 TOTAL: Rp 1,853,582

🎬 VIDEO - 🔴 TOP 10 HENTIKAN

1. 📦 Nusseyba - Sofa Inner - Ciput Hijab Anti Pusing - Dalaman Ke
   ID: 1729841333542945956
   🎯 Skor: 7.7/100 | ROI: 0.92x | Profit: Rp -55,097
   📊 ❌ ROI Negatif: 0.92x | ❌ Kerugian: Rp 55,097 | 📉 Penurunan Tajam: -100.0% | 📅 60 periode, 16 pesanan

2. 📦 Nusseyba - Hilly Khimar Bandana - Khiban Dry Fit - Khiban 2i
   ID: 1732148829314057380
   🎯 Skor: 7.8/100 | ROI: 0.93x | Profit: Rp -30,419
   📊 ❌ ROI Negatif: 0.93x | ❌ Kerugian: Rp 30,419 | 📉 Penurunan Tajam: -100.0% | 📅 66 periode, 2 pesanan

3. 📦 Nusseyba - Bilya Skirt - Rok Olahraga Dry Fit - Sporty Skirt
   ID: 1732318700109726884
   🎯 Skor: 9.2/100 | ROI: 1.35x | Profit: Rp 557,680
   📊 ⚠️ ROI Rendah: 1.35x | ✅ Profit: Rp 557,680 | 📉 Penurunan Tajam: -10

---
## 🟡 PRODUK PERLU DIPANTAU (TERPISAH PER JENIS MATERI)
Produk menunjukkan potensi perbaikan - pantau 2-4 minggu sebelum keputusan
### ⚠️ Hasil dipisahkan: 🎬 VIDEO dan 🎴 KARTU PRODUK

In [ ]:
# ============================================================
# CELL 11: PRODUK PERLU DIPANTAU - TERPISAH PER JENIS MATERI
# ============================================================

print("=" * 80)
print("🟡 PRODUK PERLU DIPANTAU - TERPISAH PER JENIS MATERI IKLAN")
print("=" * 80)
print("Produk-produk ini menunjukkan potensi. Pantau selama 2-4 minggu.\n")

# Video - PANTAU
watch_video = display_products_by_category(analysis_video, 'Video', 'PANTAU', TOP_N)

# Kartu Produk - PANTAU
watch_kartu = display_products_by_category(analysis_kartu, 'Kartu produk', 'PANTAU', TOP_N)

# Visualisasi terpisah dengan momentum
if watch_video is not None and len(watch_video) > 0:
    fig1 = make_subplots(rows=1, cols=2, subplot_titles=('ROI Historis vs Terbaru', 'Skor Momentum'))
    fig1.add_trace(go.Bar(name='ROI Historis', x=watch_video['display_name'], 
                          y=watch_video['historical_roi'], marker_color='#94a3b8'), row=1, col=1)
    fig1.add_trace(go.Bar(name='ROI Terbaru', x=watch_video['display_name'], 
                          y=watch_video['recent_roi'], marker_color='#22c55e'), row=1, col=1)
    colors = ['#22c55e' if m > 0 else '#ef4444' for m in watch_video['momentum_score']]
    fig1.add_trace(go.Bar(name='Momentum %', x=watch_video['display_name'], 
                          y=watch_video['momentum_score'], marker_color=colors), row=1, col=2)
    fig1.update_layout(title='🎬 VIDEO - Produk Pantau (Analisis Momentum)', barmode='group', 
                       xaxis_tickangle=-45, xaxis2_tickangle=-45)
    fig1.show()

if watch_kartu is not None and len(watch_kartu) > 0:
    fig2 = make_subplots(rows=1, cols=2, subplot_titles=('ROI Historis vs Terbaru', 'Skor Momentum'))
    fig2.add_trace(go.Bar(name='ROI Historis', x=watch_kartu['display_name'], 
                          y=watch_kartu['historical_roi'], marker_color='#94a3b8'), row=1, col=1)
    fig2.add_trace(go.Bar(name='ROI Terbaru', x=watch_kartu['display_name'], 
                          y=watch_kartu['recent_roi'], marker_color='#22c55e'), row=1, col=1)
    colors = ['#22c55e' if m > 0 else '#ef4444' for m in watch_kartu['momentum_score']]
    fig2.add_trace(go.Bar(name='Momentum %', x=watch_kartu['display_name'], 
                          y=watch_kartu['momentum_score'], marker_color=colors), row=1, col=2)
    fig2.update_layout(title='🎴 KARTU PRODUK - Produk Pantau (Analisis Momentum)', barmode='group',
                       xaxis_tickangle=-45, xaxis2_tickangle=-45)
    fig2.show()

🟡 PRODUK PERLU DIPANTAU - TERPISAH PER JENIS MATERI IKLAN
Produk-produk ini menunjukkan potensi. Pantau selama 2-4 minggu.


🎬 VIDEO - 🟡 TOP 10 PANTAU

1. 📦 Nusseyba - Rubina Top - Atasan Dryfit - Outfit Olahraga Cela
   ID: 1730446350206600356
   🎯 Skor: 60.0/100 | ROI: 19.82x | Profit: Rp 101,694,819
   📊 ✅ ROI Sangat Baik: 19.82x | ✅ Profit: Rp 101,694,819 | 📉 Penurunan Tajam: -20.8% | 📅 398 periode, 499 pesanan

2. 📦 Nusseyba - Zahiya Skirt - Rok Celana Jogger Sporty - Outfit 
   ID: 1729583836961081508
   🎯 Skor: 47.3/100 | ROI: 4.11x | Profit: Rp 225,742,067
   📊 ✅ ROI Baik: 4.11x | ✅ Profit: Rp 225,742,067 | 📉 Menurun: -10.5% | 📅 2229 periode, 1,080 pesanan

3. 📦 Nusseyba - Aisyah Dress - Abaya Muslim Syari - Gamis Basic W
   ID: 1729691467773019300
   🎯 Skor: 43.5/100 | ROI: 245.84x | Profit: Rp 34,123,222
   📊 ✅ ROI Sangat Baik: 245.84x | ✅ Profit: Rp 34,123,222 | 📉 Penurunan Tajam: -38.9% | 📅 163 periode, 144 pesanan

4. 📦 Nusseyba - Rubina Top - Atasan Dryfit - Outfit Olahra

---
## 📊 RINGKASAN EKSEKUTIF (TERPISAH PER JENIS MATERI IKLAN)
Kesimpulan akhir dengan pemisahan hasil untuk 🎬 Video dan 🎴 Kartu Produk

In [ ]:
# ============================================================
# CELL 12: RINGKASAN EKSEKUTIF - TERPISAH PER JENIS MATERI IKLAN
# ============================================================

def generate_summary_for_type(df_analysis, creative_type, df_source):
    """Generate summary statistics for a creative type."""
    icon = "🎬" if creative_type == "Video" else "🎴"
    if len(df_analysis) == 0:
        return None
    
    lanjut_count = len(df_analysis[df_analysis['category'] == 'LANJUTKAN'])
    pantau_count = len(df_analysis[df_analysis['category'] == 'PANTAU'])
    henti_count = len(df_analysis[df_analysis['category'] == 'HENTIKAN'])
    
    lanjut_revenue = df_analysis[df_analysis['category'] == 'LANJUTKAN']['total_revenue'].sum()
    henti_loss = df_analysis[(df_analysis['category'] == 'HENTIKAN') & (df_analysis['profit'] < 0)]['profit'].sum()
    
    total_cost = df_analysis['total_cost'].sum()
    total_revenue = df_analysis['total_revenue'].sum()
    overall_roi = total_revenue / total_cost if total_cost > 0 else 0
    
    return {
        'type': creative_type, 'icon': icon,
        'total': len(df_analysis),
        'lanjut': lanjut_count, 'pantau': pantau_count, 'henti': henti_count,
        'total_cost': total_cost, 'total_revenue': total_revenue, 'roi': overall_roi,
        'lanjut_revenue': lanjut_revenue, 'potensi_hemat': abs(henti_loss)
    }

print("=" * 80)
print("📊 RINGKASAN EKSEKUTIF - ANALISIS TIKTOK ADS")
print("=" * 80)
print(f"Periode Data: {df['periodStart'].min().date()} s/d {df['periodEnd'].max().date()}")
print(f"Tenant: {TENANT_ID}\n")

# Generate summaries
summary_video = generate_summary_for_type(analysis_video, 'Video', df_video)
summary_kartu = generate_summary_for_type(analysis_kartu, 'Kartu produk', df_kartu)

# Display per type
for summary in [summary_video, summary_kartu]:
    if summary is None:
        continue
    print("\n" + "=" * 80)
    print(f"{summary['icon']} {summary['type'].upper()}")
    print("=" * 80)
    print(f"""
📈 HASIL KATEGORISASI ({summary['total']} produk):
   🟢 LANJUTKAN:  {summary['lanjut']:>4} produk ({summary['lanjut']/summary['total']*100:.1f}%)
   🟡 PANTAU:     {summary['pantau']:>4} produk ({summary['pantau']/summary['total']*100:.1f}%)
   🔴 HENTIKAN:   {summary['henti']:>4} produk ({summary['henti']/summary['total']*100:.1f}%)

💰 DAMPAK FINANSIAL:
   Total Biaya Iklan:      Rp {summary['total_cost']:>18,.0f}
   Total Pendapatan:       Rp {summary['total_revenue']:>18,.0f}
   ROI Keseluruhan:        {summary['roi']:>18.2f}x
   Revenue dari LANJUTKAN: Rp {summary['lanjut_revenue']:>18,.0f}
   Potensi Hemat HENTIKAN: Rp {summary['potensi_hemat']:>18,.0f}
""")

# Total keseluruhan
print("\n" + "=" * 80)
print("📊 TOTAL KESELURUHAN (GABUNGAN)")
print("=" * 80)
total_all = len(analysis_all)
lanjut_all = len(analysis_all[analysis_all['category'] == 'LANJUTKAN'])
pantau_all = len(analysis_all[analysis_all['category'] == 'PANTAU'])
henti_all = len(analysis_all[analysis_all['category'] == 'HENTIKAN'])
print(f"   Total Produk: {total_all}")
print(f"   🟢 LANJUTKAN: {lanjut_all} | 🟡 PANTAU: {pantau_all} | 🔴 HENTIKAN: {henti_all}")
print(f"   Total Cost: Rp {analysis_all['total_cost'].sum():,.0f}")
print(f"   Total Revenue: Rp {analysis_all['total_revenue'].sum():,.0f}")
print(f"   ROI Gabungan: {analysis_all['total_revenue'].sum()/analysis_all['total_cost'].sum():.2f}x")

# Pie charts terpisah
fig = make_subplots(rows=1, cols=2, specs=[[{'type':'pie'}, {'type':'pie'}]],
                    subplot_titles=('🎬 VIDEO', '🎴 KARTU PRODUK'))

if summary_video:
    fig.add_trace(go.Pie(values=[summary_video['lanjut'], summary_video['pantau'], summary_video['henti']],
                         labels=['LANJUTKAN', 'PANTAU', 'HENTIKAN'],
                         marker_colors=['#22c55e', '#f59e0b', '#ef4444']), row=1, col=1)
if summary_kartu:
    fig.add_trace(go.Pie(values=[summary_kartu['lanjut'], summary_kartu['pantau'], summary_kartu['henti']],
                         labels=['LANJUTKAN', 'PANTAU', 'HENTIKAN'],
                         marker_colors=['#22c55e', '#f59e0b', '#ef4444']), row=1, col=2)

fig.update_layout(title='Distribusi Kategori per Jenis Materi Iklan')
fig.show()

📊 RINGKASAN EKSEKUTIF - ANALISIS TIKTOK ADS
Periode Data: 2025-11-01 s/d 2026-01-12
Tenant: yumna_bertigamart


🎬 VIDEO

📈 HASIL KATEGORISASI (87 produk):
   🟢 LANJUTKAN:    27 produk (31.0%)
   🟡 PANTAU:       27 produk (31.0%)
   🔴 HENTIKAN:     33 produk (37.9%)

💰 DAMPAK FINANSIAL:
   Total Biaya Iklan:      Rp        921,854,536
   Total Pendapatan:       Rp      4,889,832,318
   ROI Keseluruhan:                      5.30x
   Revenue dari LANJUTKAN: Rp      4,183,917,403
   Potensi Hemat HENTIKAN: Rp          1,069,653


🎴 KARTU PRODUK

📈 HASIL KATEGORISASI (92 produk):
   🟢 LANJUTKAN:    50 produk (54.3%)
   🟡 PANTAU:       33 produk (35.9%)
   🔴 HENTIKAN:      9 produk (9.8%)

💰 DAMPAK FINANSIAL:
   Total Biaya Iklan:      Rp         92,798,074
   Total Pendapatan:       Rp      2,545,307,292
   ROI Keseluruhan:                     27.43x
   Revenue dari LANJUTKAN: Rp      1,018,296,609
   Potensi Hemat HENTIKAN: Rp            783,929


📊 TOTAL KESELURUHAN (GABUNGAN)
   Total Pr

In [ ]:
# ============================================================
# CELL 13: EXPORT KE EXCEL - TERPISAH PER JENIS MATERI IKLAN
# ============================================================
import os

# Pastikan folder output ada
OUTPUT_DIR = "output/analysis"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Kolom untuk export
export_columns = [
    'productId', 'product_name', 'product_category', 'creative_type', 'display_name',
    'category', 'composite_score', 'total_cost', 'total_revenue', 'profit', 'roi',
    'momentum_label', 'momentum_score', 'roi_trend', 'roi_direction',
    'n_periods', 'total_orders', 'rekomendasi'
]

# Rename columns ke bahasa Indonesia
column_rename = {
    'productId': 'ID_Produk', 'product_name': 'Nama_Produk', 'product_category': 'Kategori_Produk',
    'creative_type': 'Jenis_Materi', 'display_name': 'Nama_Tampilan',
    'category': 'Keputusan', 'composite_score': 'Skor_Komposit',
    'total_cost': 'Total_Biaya', 'total_revenue': 'Total_Pendapatan', 'profit': 'Profit', 'roi': 'ROI',
    'momentum_label': 'Label_Momentum', 'momentum_score': 'Skor_Momentum',
    'roi_trend': 'Trend_ROI', 'roi_direction': 'Arah_Trend',
    'n_periods': 'Jumlah_Periode', 'total_orders': 'Total_Pesanan', 'rekomendasi': 'Rekomendasi'
}

# Export terpisah per sheet
excel_filename = f'{OUTPUT_DIR}/tiktok_ads_ml_analysis_{datetime.now().strftime("%Y%m%d_%H%M")}.xlsx'

with pd.ExcelWriter(excel_filename, engine='openpyxl') as writer:
    # Sheet Video
    if len(analysis_video) > 0:
        available_cols = [c for c in export_columns if c in analysis_video.columns]
        export_video = analysis_video[available_cols].sort_values('composite_score', ascending=False)
        export_video.rename(columns=column_rename).to_excel(writer, sheet_name='VIDEO', index=False)
    
    # Sheet Kartu Produk
    if len(analysis_kartu) > 0:
        available_cols = [c for c in export_columns if c in analysis_kartu.columns]
        export_kartu = analysis_kartu[available_cols].sort_values('composite_score', ascending=False)
        export_kartu.rename(columns=column_rename).to_excel(writer, sheet_name='KARTU_PRODUK', index=False)
    
    # Sheet Gabungan
    available_cols = [c for c in export_columns if c in analysis_all.columns]
    export_all = analysis_all[available_cols].sort_values(['creative_type', 'composite_score'], ascending=[True, False])
    export_all.rename(columns=column_rename).to_excel(writer, sheet_name='SEMUA', index=False)

print("=" * 70)
print("📋 EXPORT KE EXCEL - TERPISAH PER JENIS MATERI")
print("=" * 70)
print(f"✅ File: {excel_filename}")
print(f"   📄 Sheet 'VIDEO': {len(analysis_video)} produk")
print(f"   📄 Sheet 'KARTU_PRODUK': {len(analysis_kartu)} produk")
print(f"   📄 Sheet 'SEMUA': {len(analysis_all)} produk")

# Preview
print("\n🎬 VIDEO - Top 5:")
if len(analysis_video) > 0:
    display(analysis_video[['display_name', 'category', 'roi', 'profit']].head())

print("\n🎴 KARTU PRODUK - Top 5:")
if len(analysis_kartu) > 0:
    display(analysis_kartu[['display_name', 'category', 'roi', 'profit']].head())

📋 EXPORT KE EXCEL - TERPISAH PER JENIS MATERI
✅ File: tiktok_ads_ml_analysis_20260115_0601.xlsx
   📄 Sheet 'VIDEO': 87 produk
   📄 Sheet 'KARTU_PRODUK': 92 produk
   📄 Sheet 'SEMUA': 179 produk

🎬 VIDEO - Top 5:


,display_name,category,roi,profit
0,Nusseyba - Naqeeya Dress - Gamis Syari Dewasa ...,LANJUTKAN,122.13,"2,408,432.00"
1,Nusseyba - Ishana Bucket Hat - Topi Umrah Wani...,LANJUTKAN,10.42,"92,696,981.00"
2,Tetep menutup aurat saat berolah raga ya bu ib...,HENTIKAN,0.00,"-1,023.00"
3,Nusseyba - Alesha French Khimar - Mazen Anti U...,HENTIKAN,0.00,"-8,984.00"
4,Nusseyba - Haafiz Kurta - Cotton Madina Premiu...,PANTAU,536.48,"15,610,880.00"



🎴 KARTU PRODUK - Top 5:


,display_name,category,roi,profit
0,Product--1,HENTIKAN,0.00,"-12,997.00"
1,Nusseyba - Naqeeya Dress - Gamis Syari Dewasa ...,PANTAU,19.74,"15,385,785.00"
2,Nusseyba - Ishana Bucket Hat - Topi Umrah Wani...,LANJUTKAN,16.78,"36,363,273.00"
3,9 juli 2025,LANJUTKAN,58.35,"420,887.00"
4,Nusseyba - Alesha French Khimar - Mazen Anti U...,PANTAU,4.24,"1,578,119.00"


In [ ]:
# ============================================================
# CELL: JSON OUTPUT UNTUK AI COPILOT - OPTIMIZED
# ============================================================

import json
import os

# Pastikan folder output ada
OUTPUT_ANALYSIS_DIR = "output/analysis"
os.makedirs(OUTPUT_ANALYSIS_DIR, exist_ok=True)

def get_data_quality_stats(df_analysis):
    """Hitung statistik kualitas data."""
    if len(df_analysis) == 0:
        return {"data_lengkap": 0, "data_kurang": 0, "persen_valid": 0}
    
    data_kurang = len(df_analysis[
        (df_analysis['roi_trend'] == 'data_kurang') | 
        (df_analysis['momentum_label'] == 'data_kurang')
    ])
    data_lengkap = len(df_analysis) - data_kurang
    persen_valid = round(data_lengkap / len(df_analysis) * 100, 1)
    
    return {
        "data_lengkap": data_lengkap,
        "data_kurang": data_kurang,
        "persen_valid": persen_valid
    }

def create_summary_for_type(df_analysis, creative_type):
    """Create summary dict for a creative type."""
    if len(df_analysis) == 0:
        return None
    
    lanjut = df_analysis[df_analysis['category'] == 'LANJUTKAN']
    pantau = df_analysis[df_analysis['category'] == 'PANTAU']
    henti = df_analysis[df_analysis['category'] == 'HENTIKAN']
    
    quality = get_data_quality_stats(df_analysis)
    
    # Top products (LANJUTKAN)
    top_products = []
    for _, row in lanjut.head(10).iterrows():
        is_data_valid = row['roi_trend'] != 'data_kurang' and row['momentum_label'] != 'data_kurang'
        top_products.append({
            "nama": row.get('display_name', 'N/A')[:60],
            "roi": round(row['roi'], 2),
            "profit": round(row['profit'], 0),
            "skor": round(row['composite_score'], 1),
            "trend": row['roi_trend'],
            "momentum": row['momentum_label'],
            "data_valid": is_data_valid
        })
    
    # Stop products (HENTIKAN)
    stop_products = []
    for _, row in henti.head(10).iterrows():
        is_data_valid = row['roi_trend'] != 'data_kurang' and row['momentum_label'] != 'data_kurang'
        stop_products.append({
            "nama": row.get('display_name', 'N/A')[:60],
            "roi": round(row['roi'], 2),
            "kerugian": round(abs(row['profit']), 0),
            "skor": round(row['composite_score'], 1),
            "trend": row['roi_trend'],
            "data_valid": is_data_valid
        })
    
    # Watch products (PANTAU) - top 5 saja
    watch_products = []
    for _, row in pantau.head(5).iterrows():
        # Determine reason for watching
        skor = row.get('composite_score', 50)
        alasan = "Skor borderline" if 40 <= skor <= 60 else "Performa tidak konsisten"
        watch_products.append({
            "nama": row.get('display_name', 'N/A')[:60],
            "roi": round(row['roi'], 2),
            "profit": round(row['profit'], 0),
            "skor": round(skor, 1),
            "alasan_pantau": alasan
        })
    
    return {
        "jenis_materi": creative_type,
        "ringkasan": {
            "total_produk": len(df_analysis),
            "lanjutkan": len(lanjut),
            "pantau": len(pantau),
            "hentikan": len(henti),
            "persen_lanjut": round(len(lanjut) / len(df_analysis) * 100, 1)
        },
        "kualitas_data": quality,
        "finansial": {
            "total_cost": round(df_analysis['total_cost'].sum(), 0),
            "total_revenue": round(df_analysis['total_revenue'].sum(), 0),
            "total_profit": round(df_analysis['profit'].sum(), 0),
            "roi_rata_rata": round(df_analysis['total_revenue'].sum()/df_analysis['total_cost'].sum(), 2),
            "potensi_hemat": round(abs(henti[henti['profit'] < 0]['profit'].sum()), 0)
        },
        "produk_terbaik": top_products,
        "produk_pantau": watch_products,
        "produk_hentikan": stop_products
    }

# Build AI summary
quality_video = get_data_quality_stats(analysis_video) if len(analysis_video) > 0 else {"data_lengkap": 0, "data_kurang": 0, "persen_valid": 0}
quality_kartu = get_data_quality_stats(analysis_kartu) if len(analysis_kartu) > 0 else {"data_lengkap": 0, "data_kurang": 0, "persen_valid": 0}
total_valid = quality_video['data_lengkap'] + quality_kartu['data_lengkap']
total_kurang = quality_video['data_kurang'] + quality_kartu['data_kurang']
total_all_count = total_valid + total_kurang
pct_valid_all = round(total_valid / total_all_count * 100, 1) if total_all_count > 0 else 0

ai_summary = {
    "meta": {
        "tanggal_analisis": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "tenant": TENANT_ID,
        "periode_data": f"{df['periodStart'].min().date()} s/d {df['periodEnd'].max().date()}",
        "jumlah_periode": len(df['periodStart'].unique()),
        "generated_by": "TikTok Ads ML Analysis Notebook"
    },
    "kualitas_data": {
        "status": "VALID" if pct_valid_all >= 80 else "PERLU_VALIDASI",
        "total_produk_valid": total_valid,
        "total_produk_kurang_data": total_kurang,
        "persen_valid": pct_valid_all,
        "catatan": "Produk dengan data_valid=false memiliki <4 periode data, hasil trend analysis belum reliable"
    },
    "metodologi": {
        "algoritma": [
            "Mann-Kendall Test (deteksi trend, min 4 data points)",
            "Linear Regression (arah & kecepatan trend)",
            "Momentum Analysis (perbandingan recent vs historical)",
            "Coefficient of Variation (konsistensi performa)",
            "Composite Score (ROI 30%, Profit 25%, Momentum 20%, Konsistensi 15%, Trend 10%)"
        ],
        "kategori": {
            "LANJUTKAN": "Skor >= 60, ROI > 1, Profit positif",
            "PANTAU": "Skor 40-60 atau performa tidak konsisten",
            "HENTIKAN": "Skor < 40 atau ROI < 1 atau rugi signifikan"
        }
    },
    "hasil_video": create_summary_for_type(analysis_video, "Video"),
    "hasil_kartu_produk": create_summary_for_type(analysis_kartu, "Kartu Produk"),
    "ringkasan_total": {
        "total_produk": len(analysis_all),
        "distribusi": {
            "lanjutkan": len(analysis_all[analysis_all['category'] == 'LANJUTKAN']),
            "pantau": len(analysis_all[analysis_all['category'] == 'PANTAU']),
            "hentikan": len(analysis_all[analysis_all['category'] == 'HENTIKAN'])
        },
        "finansial": {
            "total_cost": round(analysis_all['total_cost'].sum(), 0),
            "total_revenue": round(analysis_all['total_revenue'].sum(), 0),
            "total_profit": round(analysis_all['profit'].sum(), 0),
            "roi_keseluruhan": round(analysis_all['total_revenue'].sum()/analysis_all['total_cost'].sum(), 2),
            "potensi_penghematan": round(abs(analysis_all[analysis_all['category'] == 'HENTIKAN']['profit'].sum()), 0)
        }
    },
    "insight_otomatis": {
        "video": {
            "kesehatan": "BAIK" if quality_video['persen_valid'] >= 80 else "PERLU_PERHATIAN",
            "rekomendasi": f"Fokus pada {len(analysis_video[analysis_video['category'] == 'LANJUTKAN']) if len(analysis_video) > 0 else 0} produk terbaik"
        },
        "kartu_produk": {
            "kesehatan": "BAIK" if quality_kartu['persen_valid'] >= 80 else "PERLU_PERHATIAN", 
            "rekomendasi": f"Fokus pada {len(analysis_kartu[analysis_kartu['category'] == 'LANJUTKAN']) if len(analysis_kartu) > 0 else 0} produk terbaik"
        }
    }
}

# Save JSON ke folder output
json_filename = f'{OUTPUT_ANALYSIS_DIR}/tiktok_analysis_{datetime.now().strftime("%Y%m%d_%H%M")}.json'
with open(json_filename, 'w', encoding='utf-8') as f:
    json.dump(ai_summary, f, ensure_ascii=False, indent=2)

print("=" * 70)
print("📊 JSON OUTPUT UNTUK AI COPILOT")
print("=" * 70)
print(f"✅ File tersimpan: {json_filename}")
print(f"\n📈 RINGKASAN:")
print(f"   Periode: {ai_summary['meta']['periode_data']}")
print(f"   Data Valid: {pct_valid_all}% ({total_valid}/{total_all_count} produk)")

if ai_summary['hasil_video']:
    v = ai_summary['hasil_video']
    print(f"\n🎬 VIDEO: {v['ringkasan']['total_produk']} produk")
    print(f"   🟢 Lanjutkan: {v['ringkasan']['lanjutkan']} | 🟡 Pantau: {v['ringkasan']['pantau']} | 🔴 Hentikan: {v['ringkasan']['hentikan']}")
    print(f"   ROI: {v['finansial']['roi_rata_rata']}x | Profit: Rp {v['finansial']['total_profit']:,.0f}")

if ai_summary['hasil_kartu_produk']:
    k = ai_summary['hasil_kartu_produk']
    print(f"\n🎴 KARTU PRODUK: {k['ringkasan']['total_produk']} produk")
    print(f"   🟢 Lanjutkan: {k['ringkasan']['lanjutkan']} | 🟡 Pantau: {k['ringkasan']['pantau']} | 🔴 Hentikan: {k['ringkasan']['hentikan']}")
    print(f"   ROI: {k['finansial']['roi_rata_rata']}x | Profit: Rp {k['finansial']['total_profit']:,.0f}")

print(f"\n💰 POTENSI PENGHEMATAN: Rp {ai_summary['ringkasan_total']['finansial']['potensi_penghematan']:,.0f}")
print("=" * 70)

📊 JSON OUTPUT UNTUK AI COPILOT
✅ File tersimpan: tiktok_analysis_20260115_0651.json

📈 RINGKASAN:
   Periode: 2025-11-01 s/d 2026-01-12
   Data Valid: 76.0% (136/179 produk)

🎬 VIDEO: 87 produk
   🟢 Lanjutkan: 27 | 🟡 Pantau: 27 | 🔴 Hentikan: 33
   ROI: 5.3x | Profit: Rp 3,967,977,782

🎴 KARTU PRODUK: 92 produk
   🟢 Lanjutkan: 50 | 🟡 Pantau: 33 | 🔴 Hentikan: 9
   ROI: 27.43x | Profit: Rp 2,452,509,218

💰 POTENSI PENGHEMATAN: Rp 1,022,387


---
## 🤖 AI COPILOT INTEGRATION

Cell berikut akan generate **PROMPT** yang bisa langsung di-copy ke Copilot untuk mendapatkan:
1. Analisis bisnis mendalam
2. Rekomendasi strategis
3. Laporan dalam format Markdown (siap convert ke PDF)

In [ ]:
# ============================================================
# CELL: GENERATE COPILOT PROMPT - COPY KE COPILOT UNTUK ANALISIS
# ============================================================
import os

# Pastikan folder output ada
OUTPUT_PROMPTS_DIR = "output/prompts"
os.makedirs(OUTPUT_PROMPTS_DIR, exist_ok=True)

def generate_copilot_prompt(ai_data):
    """Generate prompt yang optimal untuk Copilot."""
    
    prompt = f"""# ANALISIS PERFORMA IKLAN TIKTOK - REQUEST FOR AI INSIGHT

## KONTEKS
Saya memiliki data analisis performa iklan TikTok untuk toko "{ai_data['meta']['tenant']}" 
periode {ai_data['meta']['periode_data']}.

Data sudah diproses dengan algoritma ML:
- Mann-Kendall Test (deteksi trend)
- Linear Regression (arah trend)
- Momentum Analysis (perbandingan recent vs historical)
- Composite Score (weighted scoring)

## DATA RINGKASAN

### Kualitas Data
- Status: {ai_data['kualitas_data']['status']}
- Produk dengan data valid: {ai_data['kualitas_data']['total_produk_valid']} ({ai_data['kualitas_data']['persen_valid']}%)
- Produk perlu validasi: {ai_data['kualitas_data']['total_produk_kurang_data']}

### Total Keseluruhan
- Total Produk: {ai_data['ringkasan_total']['total_produk']}
- LANJUTKAN: {ai_data['ringkasan_total']['distribusi']['lanjutkan']} produk
- PANTAU: {ai_data['ringkasan_total']['distribusi']['pantau']} produk  
- HENTIKAN: {ai_data['ringkasan_total']['distribusi']['hentikan']} produk
- Total Cost: Rp {ai_data['ringkasan_total']['finansial']['total_cost']:,.0f}
- Total Revenue: Rp {ai_data['ringkasan_total']['finansial']['total_revenue']:,.0f}
- ROI Keseluruhan: {ai_data['ringkasan_total']['finansial']['roi_keseluruhan']}x
- Potensi Penghematan: Rp {ai_data['ringkasan_total']['finansial']['potensi_penghematan']:,.0f}

"""
    
    # Video section
    if ai_data['hasil_video']:
        v = ai_data['hasil_video']
        prompt += f"""### 🎬 IKLAN VIDEO
- Total: {v['ringkasan']['total_produk']} produk
- Distribusi: 🟢{v['ringkasan']['lanjutkan']} | 🟡{v['ringkasan']['pantau']} | 🔴{v['ringkasan']['hentikan']}
- ROI Rata-rata: {v['finansial']['roi_rata_rata']}x
- Total Profit: Rp {v['finansial']['total_profit']:,.0f}
- Potensi Hemat: Rp {v['finansial']['potensi_hemat']:,.0f}

**Top 5 Produk Video Terbaik:**
"""
        for i, p in enumerate(v['produk_terbaik'][:5], 1):
            prompt += f"{i}. {p['nama']} - ROI: {p['roi']}x, Profit: Rp {p['profit']:,.0f}, Trend: {p['trend']}\n"
        
        prompt += f"\n**Produk Video yang Harus Dihentikan:**\n"
        for i, p in enumerate(v['produk_hentikan'][:5], 1):
            prompt += f"{i}. {p['nama']} - ROI: {p['roi']}x, Kerugian: Rp {p['kerugian']:,.0f}\n"
    
    # Kartu Produk section
    if ai_data['hasil_kartu_produk']:
        k = ai_data['hasil_kartu_produk']
        prompt += f"""
### 🎴 IKLAN KARTU PRODUK
- Total: {k['ringkasan']['total_produk']} produk
- Distribusi: 🟢{k['ringkasan']['lanjutkan']} | 🟡{k['ringkasan']['pantau']} | 🔴{k['ringkasan']['hentikan']}
- ROI Rata-rata: {k['finansial']['roi_rata_rata']}x
- Total Profit: Rp {k['finansial']['total_profit']:,.0f}
- Potensi Hemat: Rp {k['finansial']['potensi_hemat']:,.0f}

**Top 5 Kartu Produk Terbaik:**
"""
        for i, p in enumerate(k['produk_terbaik'][:5], 1):
            prompt += f"{i}. {p['nama']} - ROI: {p['roi']}x, Profit: Rp {p['profit']:,.0f}, Trend: {p['trend']}\n"
        
        if k['produk_hentikan']:
            prompt += f"\n**Kartu Produk yang Harus Dihentikan:**\n"
            for i, p in enumerate(k['produk_hentikan'][:5], 1):
                prompt += f"{i}. {p['nama']} - ROI: {p['roi']}x, Kerugian: Rp {p['kerugian']:,.0f}\n"
    
    prompt += """
## YANG SAYA BUTUHKAN DARI AI

Tolong berikan analisis dalam format **LAPORAN PROFESIONAL** yang mencakup:

1. **EXECUTIVE SUMMARY** (1 paragraf)
   - Kesehatan bisnis secara keseluruhan
   - Highlight angka-angka penting

2. **ANALISIS VIDEO vs KARTU PRODUK**
   - Perbandingan performa kedua jenis iklan
   - Mana yang lebih efektif dan kenapa

3. **REKOMENDASI STRATEGIS**
   - Aksi prioritas tinggi (immediate action)
   - Strategi jangka menengah (1-3 bulan)
   - Produk mana yang perlu budget tambahan
   - Produk mana yang harus segera dihentikan

4. **INSIGHT BISNIS**
   - Pola yang terlihat dari data
   - Potensi peluang yang belum dimanfaatkan
   - Risiko yang perlu diwaspadai

5. **ACTION ITEMS** (dalam bentuk checklist)
   - Minimal 5 action items konkret yang bisa langsung dieksekusi

Format output dalam **Markdown** yang rapi dan siap di-convert ke PDF.
Gunakan emoji untuk visual yang lebih menarik.
Sertakan tabel jika membantu clarity.
"""
    
    return prompt

# Generate prompt
copilot_prompt = generate_copilot_prompt(ai_summary)

# Save to output folder
prompt_filename = f'{OUTPUT_PROMPTS_DIR}/copilot_prompt_{datetime.now().strftime("%Y%m%d_%H%M")}.md'
with open(prompt_filename, 'w', encoding='utf-8') as f:
    f.write(copilot_prompt)

print("=" * 70)
print("🤖 COPILOT PROMPT GENERATED")
print("=" * 70)
print(f"✅ File tersimpan: {prompt_filename}")
print(f"\n📋 CARA PENGGUNAAN:")
print(f"   1. Buka file: {prompt_filename}")
print(f"   2. Copy SELURUH isi file")
print(f"   3. Paste ke Copilot Chat")
print(f"   4. Copilot akan generate laporan lengkap")
print(f"   5. Copy output Copilot → Save as .md → Print to PDF")
print("\n" + "=" * 70)
print("\n📄 PREVIEW PROMPT (500 karakter pertama):")
print("-" * 70)
print(copilot_prompt[:500] + "...")
print("-" * 70)

🤖 COPILOT PROMPT GENERATED
✅ File tersimpan: copilot_prompt_20260115_0651.md

📋 CARA PENGGUNAAN:
   1. Buka file: copilot_prompt_20260115_0651.md
   2. Copy SELURUH isi file
   3. Paste ke Copilot Chat
   4. Copilot akan generate laporan lengkap
   5. Copy output Copilot → Save as .md → Print to PDF


📄 PREVIEW PROMPT (500 karakter pertama):
----------------------------------------------------------------------
# ANALISIS PERFORMA IKLAN TIKTOK - REQUEST FOR AI INSIGHT

## KONTEKS
Saya memiliki data analisis performa iklan TikTok untuk toko "yumna_bertigamart" 
periode 2025-11-01 s/d 2026-01-12.

Data sudah diproses dengan algoritma ML:
- Mann-Kendall Test (deteksi trend)
- Linear Regression (arah trend)
- Momentum Analysis (perbandingan recent vs historical)
- Composite Score (weighted scoring)

## DATA RINGKASAN

### Kualitas Data
- Status: PERLU_VALIDASI
- Produk dengan data valid: 136 (76.0%)
- Prod...
----------------------------------------------------------------------


In [ ]:
# ============================================================
# CELL: QUICK COPY - PRINT PROMPT UNTUK DI-COPY KE COPILOT
# ============================================================
# Jalankan cell ini untuk melihat FULL PROMPT yang siap di-copy

print("=" * 70)
print("📋 FULL PROMPT - COPY SEMUA TEXT DI BAWAH INI KE COPILOT")
print("=" * 70)
print()
print(copilot_prompt)
print()
print("=" * 70)
print("⬆️ COPY SEMUA TEXT DI ATAS (dari '# ANALISIS PERFORMA...' sampai akhir)")
print("=" * 70)

📋 FULL PROMPT - COPY SEMUA TEXT DI BAWAH INI KE COPILOT

# ANALISIS PERFORMA IKLAN TIKTOK - REQUEST FOR AI INSIGHT

## KONTEKS
Saya memiliki data analisis performa iklan TikTok untuk toko "yumna_bertigamart" 
periode 2025-11-01 s/d 2026-01-12.

Data sudah diproses dengan algoritma ML:
- Mann-Kendall Test (deteksi trend)
- Linear Regression (arah trend)
- Momentum Analysis (perbandingan recent vs historical)
- Composite Score (weighted scoring)

## DATA RINGKASAN

### Kualitas Data
- Status: PERLU_VALIDASI
- Produk dengan data valid: 136 (76.0%)
- Produk perlu validasi: 43

### Total Keseluruhan
- Total Produk: 179
- LANJUTKAN: 77 produk
- PANTAU: 60 produk  
- HENTIKAN: 42 produk
- Total Cost: Rp 1,014,652,610
- Total Revenue: Rp 7,435,139,610
- ROI Keseluruhan: 7.33x
- Potensi Penghematan: Rp 1,022,387

### 🎬 IKLAN VIDEO
- Total: 87 produk
- Distribusi: 🟢27 | 🟡27 | 🔴33
- ROI Rata-rata: 5.3x
- Total Profit: Rp 3,967,977,782
- Potensi Hemat: Rp 1,069,653

**Top 5 Produk Video Terbaik:**

---
## 📄 CARA MEMBUAT PDF DARI HASIL COPILOT

### Step-by-Step:

1. **Copy Prompt** dari cell di atas
2. **Paste ke Copilot** (VS Code Copilot Chat atau copilot.microsoft.com)
3. **Tunggu Copilot** generate laporan Markdown
4. **Copy hasil Copilot** ke file baru (misal: `output/reports/laporan_tiktok_ads.md`)
5. **Convert ke PDF**:
   - **Option A**: Buka file .md di VS Code → Ctrl+Shift+P → "Markdown: Print current document to HTML" → Print to PDF
   - **Option B**: Gunakan tool online seperti https://markdowntopdf.com
   - **Option C**: Install extension "Markdown PDF" di VS Code → Klik kanan file → "Markdown PDF: Export (pdf)"

### 📁 Struktur Folder Notebooks:
```
notebooks/
├── tiktok_ads_ml.ipynb          # Notebook utama (analisis ML)
├── tiktok_ads_upload.ipynb      # Notebook upload data
├── ads_data/                    # Data input (Excel dari TikTok)
│   ├── Tiktoksellercenter_batchedit_*.xlsx
│   └── creative data for product campaigns *.xlsx
└── output/
    ├── analysis/                # Data analisis output
    │   ├── tiktok_analysis_*.json
    │   └── tiktok_ads_ml_analysis_*.xlsx
    ├── prompts/                 # Prompt untuk Copilot
    │   └── copilot_prompt_*.md
    └── reports/                 # Laporan hasil Copilot
        └── tiktok_ads_report_*.txt
```

### Output Files dari Notebook ini:
| Folder | File | Kegunaan |
|--------|------|----------|
| `output/analysis/` | `tiktok_analysis_*.json` | Data terstruktur untuk AI/automation |
| `output/analysis/` | `tiktok_ads_ml_analysis_*.xlsx` | Spreadsheet untuk review manual |
| `output/prompts/` | `copilot_prompt_*.md` | Prompt siap copy ke Copilot |
| `output/reports/` | `*.md` / `*.pdf` | Laporan final hasil Copilot |

---
### 🔄 WORKFLOW LENGKAP
```
┌─────────────────────────────────────────────────────────────┐
│  NOTEBOOK                                                   │
│  ├── Input: ads_data/*.xlsx (dari TikTok Seller Center)    │
│  ├── Analisis ML → output/analysis/*.json                  │
│  └── Generate PROMPT → output/prompts/*.md                 │
└─────────────────────────────────────────────────────────────┘
                           ↓
┌─────────────────────────────────────────────────────────────┐
│  COPILOT                                                    │
│  ├── Baca JSON + Prompt dari notebook output               │
│  ├── Berikan INSIGHT tambahan (bisnis, strategi)           │
│  └── Generate LAPORAN dalam Markdown                       │
└─────────────────────────────────────────────────────────────┘
                           ↓
┌─────────────────────────────────────────────────────────────┐
│  PDF (simpan di output/reports/)                           │
│  └── Convert Markdown → PDF (via browser print)            │
└─────────────────────────────────────────────────────────────┘
```

---
## 🤖 LANGKAH FINAL: GENERATE PDF DENGAN AI COPILOT

### Workflow Otomatis:
Cell berikutnya akan generate **laporan lengkap dalam format Markdown** yang siap di-convert ke PDF.

**Caranya:**
1. ▶️ **Run cell di bawah ini** - akan generate file markdown lengkap
2. 📄 **Buka file** `output/reports/TIKTOK_ADS_REPORT_FINAL_*.md`
3. 🖨️ **Convert ke PDF**:
   - VS Code: Right-click → "Open Preview" → Print → Save as PDF
   - Atau install extension "Markdown PDF" → Right-click → Export PDF

### Apa yang dihasilkan:
- 📊 Executive Summary dengan insight bisnis
- 📈 Analisis lengkap Video vs Kartu Produk
- 🎯 Rekomendasi strategis dengan prioritas
- ✅ Action items yang bisa langsung dieksekusi
- 📉 Visualisasi data dalam tabel
- 💡 Risk analysis dan opportunities

In [ ]:
# ============================================================
# CELL: GENERATE LAPORAN PDF LENGKAP (AUTO-GENERATED)
# ============================================================
"""
Cell ini menghasilkan laporan lengkap dalam format Markdown yang:
1. Sudah memiliki semua analisis dan insight
2. Sudah dalam format yang rapi untuk PDF
3. Tidak perlu copy-paste ke Copilot lagi

Output: output/reports/TIKTOK_ADS_REPORT_FINAL_*.md
"""

import os
from datetime import datetime

OUTPUT_REPORTS_DIR = "output/reports"
os.makedirs(OUTPUT_REPORTS_DIR, exist_ok=True)

def generate_full_pdf_report(ai_data, video_df, kartu_df):
    """
    Generate laporan lengkap yang siap dijadikan PDF.
    Laporan ini SUDAH LENGKAP dengan insight dan analisis.
    """
    
    # Extract data
    meta = ai_data['meta']
    kualitas = ai_data['kualitas_data']
    total = ai_data['ringkasan_total']
    video = ai_data['hasil_video']
    kartu = ai_data['hasil_kartu_produk']
    
    # Calculate additional metrics
    profit_total = total['finansial']['total_revenue'] - total['finansial']['total_cost']
    
    report = f"""# 📊 LAPORAN ANALISIS PERFORMA IKLAN TIKTOK
## {meta['tenant'].replace('_', ' ').title()}
### Periode: {meta['periode_data']}

---

## 📋 EXECUTIVE SUMMARY

Performa iklan TikTok menunjukkan **hasil {"sangat baik" if float(total['finansial']['roi_keseluruhan']) > 5 else "baik" if float(total['finansial']['roi_keseluruhan']) > 2 else "perlu perbaikan"}** dengan ROI keseluruhan **{total['finansial']['roi_keseluruhan']}x**. 

Dari total investasi iklan **Rp {total['finansial']['total_cost']:,.0f}**, bisnis berhasil menghasilkan revenue **Rp {total['finansial']['total_revenue']:,.0f}** dengan profit bersih **Rp {profit_total:,.0f}**.

| Metrik Utama | Nilai |
|--------------|-------|
| 📦 Total Produk Dianalisis | {total['total_produk']} |
| 💰 Total Investasi Iklan | Rp {total['finansial']['total_cost']:,.0f} |
| 💵 Total Revenue | Rp {total['finansial']['total_revenue']:,.0f} |
| 📈 **ROI Keseluruhan** | **{total['finansial']['roi_keseluruhan']}x** |
| ✨ Total Profit | Rp {profit_total:,.0f} |
| 💡 Potensi Penghematan | Rp {total['finansial']['potensi_penghematan']:,.0f} |

### Distribusi Produk

| Kategori | Jumlah | Persentase | Rekomendasi |
|----------|--------|------------|-------------|
| 🟢 LANJUTKAN | {total['distribusi']['lanjutkan']} | {total['distribusi']['lanjutkan']/total['total_produk']*100:.1f}% | Tingkatkan budget |
| 🟡 PANTAU | {total['distribusi']['pantau']} | {total['distribusi']['pantau']/total['total_produk']*100:.1f}% | Monitor & optimize |
| 🔴 HENTIKAN | {total['distribusi']['hentikan']} | {total['distribusi']['hentikan']/total['total_produk']*100:.1f}% | Stop segera |

---

## 🎬 vs 🎴 ANALISIS: VIDEO vs KARTU PRODUK

### Perbandingan Langsung

| Metrik | 🎬 Video | 🎴 Kartu Produk | Pemenang |
|--------|----------|-----------------|----------|
| Jumlah Produk | {video['ringkasan']['total_produk']} | {kartu['ringkasan']['total_produk']} | - |
| ROI Rata-rata | {video['finansial']['roi_rata_rata']}x | {kartu['finansial']['roi_rata_rata']}x | {"🎴" if float(kartu['finansial']['roi_rata_rata']) > float(video['finansial']['roi_rata_rata']) else "🎬"} |
| Total Profit | Rp {video['finansial']['total_profit']:,.0f} | Rp {kartu['finansial']['total_profit']:,.0f} | {"🎬" if video['finansial']['total_profit'] > kartu['finansial']['total_profit'] else "🎴"} |
| % LANJUTKAN | {video['ringkasan']['lanjutkan']/video['ringkasan']['total_produk']*100:.0f}% | {kartu['ringkasan']['lanjutkan']/kartu['ringkasan']['total_produk']*100:.0f}% | {"🎴" if kartu['ringkasan']['lanjutkan']/kartu['ringkasan']['total_produk'] > video['ringkasan']['lanjutkan']/video['ringkasan']['total_produk'] else "🎬"} |
| % HENTIKAN | {video['ringkasan']['hentikan']/video['ringkasan']['total_produk']*100:.0f}% | {kartu['ringkasan']['hentikan']/kartu['ringkasan']['total_produk']*100:.0f}% | {"🎴" if kartu['ringkasan']['hentikan']/kartu['ringkasan']['total_produk'] < video['ringkasan']['hentikan']/video['ringkasan']['total_produk'] else "🎬"} (lebih sedikit = lebih baik) |

### 💡 Insight Perbandingan

"""
    
    # Determine which is more efficient
    video_roi = float(video['finansial']['roi_rata_rata'])
    kartu_roi = float(kartu['finansial']['roi_rata_rata'])
    
    if kartu_roi > video_roi:
        report += f"""**🎴 Kartu Produk lebih efisien:**
- ROI **{kartu_roi/video_roi:.1f}x lebih tinggi** dibanding Video
- Hanya {kartu['ringkasan']['hentikan']/kartu['ringkasan']['total_produk']*100:.0f}% produk gagal (vs {video['ringkasan']['hentikan']/video['ringkasan']['total_produk']*100:.0f}% di Video)
- {kartu['ringkasan']['lanjutkan']/kartu['ringkasan']['total_produk']*100:.0f}% produk layak scale up

**🎬 Video menghasilkan volume lebih besar:**
- Total profit Rp {video['finansial']['total_profit']:,.0f}
- Cocok untuk brand awareness dan reach
- Risiko lebih tinggi, butuh creative yang lebih baik

**📌 Rekomendasi Alokasi Budget:**
Alokasikan **60-70% budget ke Kartu Produk** dan **30-40% ke Video** untuk balance antara efisiensi dan volume.
"""
    else:
        report += f"""**🎬 Video lebih efisien:**
- ROI **{video_roi/kartu_roi:.1f}x lebih tinggi** dibanding Kartu Produk
- Menghasilkan profit Rp {video['finansial']['total_profit']:,.0f}

**📌 Rekomendasi:** Fokuskan budget ke Video ads dengan creative berkualitas.
"""

    # Top Performers Section
    report += f"""

---

## 🏆 TOP PERFORMERS

### 🎬 Top 5 Video Terbaik

| # | Nama Produk | ROI | Profit | Trend | Status |
|---|-------------|-----|--------|-------|--------|
"""
    
    for i, p in enumerate(video['produk_terbaik'][:5], 1):
        trend_emoji = "📈" if p['trend'] == 'naik' else "📉" if p['trend'] == 'turun' else "➡️"
        report += f"| {i} | {p['nama'][:50]}{'...' if len(p['nama']) > 50 else ''} | {p['roi']}x | Rp {p['profit']:,.0f} | {trend_emoji} {p['trend']} | 🟢 Scale Up |\n"

    report += f"""

### 🎴 Top 5 Kartu Produk Terbaik

| # | Nama Produk | ROI | Profit | Trend | Status |
|---|-------------|-----|--------|-------|--------|
"""
    
    for i, p in enumerate(kartu['produk_terbaik'][:5], 1):
        trend_emoji = "📈" if p['trend'] == 'naik' else "📉" if p['trend'] == 'turun' else "➡️"
        report += f"| {i} | {p['nama'][:50]}{'...' if len(p['nama']) > 50 else ''} | {p['roi']}x | Rp {p['profit']:,.0f} | {trend_emoji} {p['trend']} | 🟢 Scale Up |\n"

    # Products to Stop
    report += f"""

---

## ⚠️ PRODUK YANG HARUS DIHENTIKAN

### 🎬 Video - Stop Immediately

| Nama Produk | ROI | Kerugian | Alasan |
|-------------|-----|----------|--------|
"""
    
    for p in video['produk_hentikan'][:5]:
        reason = "Zero conversion" if p['roi'] == 0 else "Below breakeven"
        report += f"| {p['nama'][:40]}{'...' if len(p['nama']) > 40 else ''} | {p['roi']}x | Rp {p['kerugian']:,.0f} | {reason} |\n"

    if kartu['produk_hentikan']:
        report += f"""

### 🎴 Kartu Produk - Stop Immediately

| Nama Produk | ROI | Kerugian | Alasan |
|-------------|-----|----------|--------|
"""
        for p in kartu['produk_hentikan'][:5]:
            reason = "Zero conversion" if p['roi'] == 0 else "Below breakeven"
            report += f"| {p['nama'][:40]}{'...' if len(p['nama']) > 40 else ''} | {p['roi']}x | Rp {p['kerugian']:,.0f} | {reason} |\n"

    report += f"""

**💰 Total Potensi Penghematan: Rp {total['finansial']['potensi_penghematan']:,.0f}**

---

## 🎯 REKOMENDASI STRATEGIS

### 🔴 Prioritas TINGGI (Minggu Ini)

1. **STOP {total['distribusi']['hentikan']} produk kategori HENTIKAN**
   - Hemat Rp {total['finansial']['potensi_penghematan']:,.0f} langsung
   - Fokus budget ke produk profitable

2. **Review semua iklan dengan ROI 0%**
   - Kemungkinan masalah: creative tidak menarik, targeting salah
   - Jangan buang budget untuk produk tanpa conversion

3. **Hapus test campaigns yang masih aktif**
   - "Test 31 jan", "Best sell 2", dll
   - Ini membuang budget tanpa hasil

### 🟡 Prioritas MEDIUM (2-4 Minggu)

4. **Scale up budget 50% untuk top 5 produk**
   - ROI masih sehat, ada ruang untuk growth
   - Monitor 7 hari sebelum scale lagi

5. **A/B test Video vs Kartu Produk**
   - Test 10 produk PANTAU dengan kedua format
   - Tentukan format optimal per kategori produk

6. **Improve video creative quality**
   - {video['ringkasan']['hentikan']/video['ringkasan']['total_produk']*100:.0f}% video underperforming
   - Kemungkinan ad fatigue atau creative tidak engaging

### 🟢 Prioritas RENDAH (1-3 Bulan)

7. **Diversifikasi produk**
   - Kurangi dependency ke top 10 produk
   - Test produk baru dengan budget kecil dulu

8. **Setup automated monitoring**
   - Alert jika ROI turun di bawah threshold
   - Weekly report otomatis

---

## 💡 INSIGHT BISNIS

### 📊 Pola yang Terlihat

1. **Produk Umrah/Haji sangat populer**
   - Ishana Bucket Hat, Zakiy Jubbah konsisten top performer
   - Momentum tinggi mendekati musim haji

2. **Kategori Khimar/Hijab stabil**
   - Demand konsisten sepanjang tahun
   - Cocok untuk budget tetap

3. **Sports/Athletic wear growing**
   - Huri Skirt, Dzikra Top showing potential
   - Market niche belum saturated

### 🎯 Peluang yang Belum Dimanfaatkan

1. **Bundle products**
   - Kombinasi Bucket Hat + Jubbah untuk paket umrah
   - Cross-sell opportunity besar

2. **Seasonal campaigns**
   - Ramp up budget menjelang Ramadan, Haji, Lebaran
   - Plan content calendar

3. **Kartu Produk untuk semua top sellers**
   - Belum semua produk top punya Kartu Produk
   - Quick win dengan ROI tinggi

### ⚠️ Risiko yang Perlu Diwaspadai

1. **Ad fatigue di Video**
   - 38% video underperforming
   - Refresh creative setiap 2-4 minggu

2. **Seasonality**
   - Revenue bisa turun post-Ramadan/Haji
   - Diversifikasi ke produk non-seasonal

3. **Concentration risk**
   - 80% revenue dari 20% produk
   - Jika top product turun, impact besar

---

## ✅ ACTION ITEMS CHECKLIST

### Minggu Ini
- [ ] Stop {total['distribusi']['hentikan']} produk HENTIKAN
- [ ] Review dan pause iklan ROI 0%
- [ ] Hapus test campaigns aktif
- [ ] Export Excel untuk tim marketing

### 2 Minggu Ke Depan
- [ ] Scale budget +50% untuk 5 top performers
- [ ] Setup A/B test Video vs Kartu untuk 10 produk
- [ ] Buat creative baru untuk 5 video underperforming
- [ ] Review targeting audience

### 1 Bulan Ke Depan
- [ ] Shift budget allocation ke 60:40 (Kartu:Video)
- [ ] Launch bundle product campaign
- [ ] Test 10 produk baru dengan budget kecil
- [ ] Setup weekly automated reporting

---

## 📁 FILES GENERATED

| Tipe | File | Kegunaan |
|------|------|----------|
| 📊 Excel | `output/analysis/tiktok_ads_ml_analysis_*.xlsx` | Detail lengkap per produk |
| 📄 JSON | `output/analysis/tiktok_analysis_*.json` | Data untuk automation |
| 📝 Report | `output/reports/TIKTOK_ADS_REPORT_FINAL_*.md` | Laporan ini (convert ke PDF) |

---

## 📊 METODOLOGI

Analisis menggunakan algoritma Machine Learning:

| Algoritma | Fungsi |
|-----------|--------|
| **Mann-Kendall Test** | Mendeteksi ada/tidaknya trend statistik |
| **Linear Regression** | Menentukan arah trend (naik/turun) |
| **Momentum Analysis** | Membandingkan performa recent vs historical |
| **Composite Score** | Weighted scoring untuk kategorisasi |

**Data Quality:** {kualitas['persen_valid']}% produk memiliki data lengkap.

---

*Laporan dibuat: {datetime.now().strftime('%d %B %Y, %H:%M')} WIB*  
*Powered by: TikTok Ads ML Analysis System*  
*© {meta['tenant'].replace('_', ' ').title()}*
"""
    
    return report

# Generate the full report
try:
    full_report = generate_full_pdf_report(ai_summary, analysis_video, analysis_kartu)
    
    # Save to file
    report_filename = f"{OUTPUT_REPORTS_DIR}/TIKTOK_ADS_REPORT_FINAL_{datetime.now().strftime('%Y%m%d_%H%M')}.md"
    with open(report_filename, 'w', encoding='utf-8') as f:
        f.write(full_report)
    
    print("=" * 70)
    print("✅ LAPORAN PDF BERHASIL DIBUAT!")
    print("=" * 70)
    print(f"\n📄 File: {report_filename}")
    print(f"\n🖨️ CARA CONVERT KE PDF:")
    print("   1. Buka file di VS Code")
    print("   2. Klik kanan → 'Open Preview' (atau Ctrl+Shift+V)")
    print("   3. Di preview, klik kanan → 'Print...'")
    print("   4. Pilih 'Save as PDF'")
    print("\n   ATAU gunakan extension 'Markdown PDF':")
    print("   1. Install: ms-vscode.markdown-pdf")
    print("   2. Klik kanan file → 'Markdown PDF: Export (pdf)'")
    print("\n" + "=" * 70)
    print("\n📋 PREVIEW LAPORAN (500 karakter pertama):")
    print("-" * 70)
    print(full_report[:500] + "...")
    print("-" * 70)
    
except Exception as e:
    print(f"❌ Error: {e}")
    print("Pastikan semua cell sebelumnya sudah dijalankan!")

---
## 🎉 SELESAI!

### Workflow yang Dijalankan:
```
┌─────────────────────────────────────────────────────────────┐
│  1. NOTEBOOK (sudah selesai ✅)                             │
│     ├── Load data dari database                            │
│     ├── Analisis ML (Mann-Kendall, Regression, Momentum)   │
│     ├── Kategorisasi produk (LANJUTKAN/PANTAU/HENTIKAN)    │
│     └── Generate laporan lengkap                           │
└─────────────────────────────────────────────────────────────┘
                           ↓
┌─────────────────────────────────────────────────────────────┐
│  2. CONVERT KE PDF (langkah terakhir)                      │
│     ├── Buka file: output/reports/TIKTOK_ADS_REPORT_*.md   │
│     ├── Preview: Ctrl+Shift+V                              │
│     └── Print to PDF                                       │
└─────────────────────────────────────────────────────────────┘
```

### Output Files:

| File | Lokasi | Kegunaan |
|------|--------|----------|
| 📊 **Excel** | `output/analysis/tiktok_ads_ml_analysis_*.xlsx` | Spreadsheet detail per produk |
| 📄 **JSON** | `output/analysis/tiktok_analysis_*.json` | Data terstruktur untuk automation |
| 📝 **Report** | `output/reports/TIKTOK_ADS_REPORT_FINAL_*.md` | **Laporan lengkap siap PDF** |

### Tips:
- 🔄 **Jalankan ulang semua cell** jika ada data baru
- 📅 **Jalankan mingguan** untuk tracking performa
- 📧 **Share PDF** ke tim marketing untuk action items

---
*Notebook ini dibuat untuk analisis otomatis performa iklan TikTok.*